# Extract variables from ADAPTS MRT dataset

In [286]:
# 0. import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
%matplotlib inline
from matplotlib.ticker import MaxNLocator
import matplotlib.dates as mdates
import datetime
import json
from pathlib import Path
import re

In [287]:
# read ProjectDeviceData_selected_fields_combined.csv
folder = Path("/Users/xueqingliu/Harvard University Dropbox/Liu Xueqing/ADAPT_MRT/rawdata/_combined")

project_device_data = pd.read_csv(folder / "ProjectDeviceData_selected_fields_combined.csv")

In [288]:

# -----------------------------
# 1) Build participant-date -> timezone lookup
# -----------------------------
# project_device_data should already contain:
# ParticipantIdentifier (or participantidentifier), date, timeZone, utcOffset

tz_lookup = (
    project_device_data.rename(columns={"participantidentifier": "ParticipantIdentifier"})
    .copy()
)

# keep only needed columns
tz_lookup = tz_lookup[["ParticipantIdentifier", "date", "timeZone", "utcOffset"]].copy()

# normalize date dtype
tz_lookup["date"] = pd.to_datetime(tz_lookup["date"], errors="coerce").dt.date

# pick one timezone record per participant/date (first non-null values)
tz_lookup = (
    tz_lookup.sort_values(["ParticipantIdentifier", "date"])
             .groupby(["ParticipantIdentifier", "date"], as_index=False)
             .agg({
                 "timeZone": lambda s: next((x for x in s if pd.notna(x) and str(x).strip() != ""), np.nan),
                 "utcOffset": lambda s: next((x for x in s if pd.notna(x) and str(x).strip() != ""), np.nan),
             })
)

# -----------------------------
# 2) Helpers: convert UTC -> local per row
# -----------------------------
def _offset_to_timedelta(offset_str):
    # supports formats like -07:00:00 or +05:30:00
    if pd.isna(offset_str):
        return pd.NaT
    m = re.match(r"^([+-])(\d{2}):(\d{2})(?::(\d{2}))?$", str(offset_str).strip())
    if not m:
        return pd.NaT
    sign = -1 if m.group(1) == "-" else 1
    hh = int(m.group(2))
    mm = int(m.group(3))
    ss = int(m.group(4) or 0)
    return sign * pd.Timedelta(hours=hh, minutes=mm, seconds=ss)

def _convert_one_utc_to_local(ts_utc, tz_name, utc_offset):
    if pd.isna(ts_utc):
        return pd.NaT

    ts_utc = pd.to_datetime(ts_utc, errors="coerce", utc=True)
    if pd.isna(ts_utc):
        return pd.NaT

    # Preferred: IANA timezone (handles DST correctly)
    if pd.notna(tz_name) and str(tz_name).strip() != "":
        try:
            return ts_utc.tz_convert(str(tz_name)).tz_localize(None)
        except Exception:
            pass

    # Fallback: static UTC offset (no DST logic)
    delta = _offset_to_timedelta(utc_offset)
    if pd.isna(delta):
        return ts_utc.tz_localize(None)  # fallback to UTC naive
    return (ts_utc + delta).tz_localize(None)

def convert_utc_columns_to_user_local(
    df,
    datetime_cols,
    participant_col="ParticipantIdentifier",
    join_date_col="InsertedDate",  # use "Timestamp" for pageview
):
    out = df.copy()

    out["_join_date"] = (
        pd.to_datetime(out[join_date_col], errors="coerce", utc=True).dt.date
    )

    out = out.merge(
        tz_lookup,
        how="left",
        left_on=[participant_col, "_join_date"],
        right_on=["ParticipantIdentifier", "date"],
        suffixes=("", "_tz"),
    )

    for col in datetime_cols:
        out[col] = [
            _convert_one_utc_to_local(ts, tz, off)
            for ts, tz, off in zip(out[col], out["timeZone"], out["utcOffset"])
        ]

    out = out.drop(
        columns=["_join_date", "ParticipantIdentifier_tz", "date", "timeZone", "utcOffset"],
        errors="ignore",
    )
    return out


In [289]:
# remove testers from all data
testers = pd.read_csv(
    folder / "Testers.csv"
)

testers_id = testers.ParticipantIdentifier.values

# add 'test-Yuxuan' to the testers_id
testers_id = np.concatenate([testers_id, ['test-Yuxuan']])

all_participant_ids = project_device_data.participantidentifier.unique()

print(all_participant_ids)

real_participant_ids = all_participant_ids[~np.isin(all_participant_ids, testers_id)]

print(real_participant_ids)



['test-Steven' '72' '128' 'ADAPT-4797-3765' 'test-Jose' '151' '106' '82'
 'test-Xueqing' 'test-Susan' '112' '143' '75' 'test-Mark' 'test-Pedja'
 '138' 'ADAPT-0565-3893' '117' '129' 'test-Daiqi' '86' '37' '18' '33' '31'
 '22' '118' 'test-Chelsie' 'test-Soo' '170' '156' '141' '184' '204' '188'
 '160' '219' '225' '195' '251' '252' '257' '239' 'test-andrew'
 'test-wenching' '248' '277' '291' 'test-Yuxuan' '285' '299' '259']
['72' '128' '151' '106' '82' '112' '143' '75' '138' '117' '129' '86' '37'
 '18' '33' '31' '22' '118' '170' '156' '141' '184' '204' '188' '160' '219'
 '225' '195' '251' '252' '257' '239' '248' '277' '291' '285' '299' '259']


In [290]:

# survey task and survey question are linked by surveykey (surveytask.surveykey = surveyquestion.surveykey)
# but this does not distinguish between different days and different participants and different questions
# participantidentifier
# resultidentifier
# answers
# question_startdate
# question_enddate

surveytask = pd.read_csv(
    folder / "SurveyTasks.csv"
)
surveyquestionresults = pd.read_csv(
    folder / "SurveyResults.csv"
)

# add a date column to the dataframe and remove duplicate rows
surveytask["date"] = pd.to_datetime(surveytask["InsertedDate"]).dt.date

surveyquestionresults["date"] = pd.to_datetime(surveyquestionresults["InsertedDate"]).dt.date

# surveytask
# surveytask = convert_utc_columns_to_user_local(
#     surveytask,
#     datetime_cols=["InsertedDate", "DueDate"],
#     participant_col="ParticipantIdentifier"
# )

# surveyquestionresults (adjust participant_col if column name differs)
# surveyquestionresults = convert_utc_columns_to_user_local(
#     surveyquestionresults,
#     datetime_cols=["StartDate", "EndDate", "InsertedDate"],  # keep only columns that exist
#     participant_col="ParticipantIdentifier"
# )

survey_key_weekly = surveytask[surveytask.SurveyName == 'MRT - Weekly Check-in survey'].SurveyKey.values[0]
survey_key_monthly = surveytask[surveytask.SurveyName == 'MRT - Monthly check-in survey and goal setting'].SurveyKey.values[0]
survey_key_last_monthly = surveytask[surveytask.SurveyName == 'MRT - Monthly check-in survey (Final month)'].SurveyKey.values[0]
survey_key_daily = surveytask[surveytask.SurveyName == 'MRT - Daily End of day survey and Planning Exercise'].SurveyKey.values[0]

# transform survey keys to lowercase to match the surveyquestionresults.surveykey
survey_key_weekly = survey_key_weekly.lower()
survey_key_monthly = survey_key_monthly.lower()
survey_key_last_monthly = survey_key_last_monthly.lower()
survey_key_daily = survey_key_daily.lower()

print(survey_key_weekly, survey_key_monthly, survey_key_last_monthly, survey_key_daily)

# remove testers from surveytask and surveyquestionresults
surveytask = surveytask[~surveytask.ParticipantIdentifier.isin(testers_id)]
surveyquestionresults = surveyquestionresults[~surveyquestionresults.ParticipantIdentifier.isin(testers_id)]

print(surveyquestionresults.head())
# check whether the record of step count start date is the same as the baseline survey date

17fb11e8-add3-ef11-bafd-0affe8024cad 9015c191-c531-f011-ad1f-0e0e6a19462f c72f18a8-c931-f011-ad1f-0e0e6a19462f a98012db-67a3-ef11-bafb-0affe8024cad
  DeviceName DeviceOSVersion DevicePlatform                    EndDate  \
0        NaN             NaN      WebClient  2025-09-12T08:04:43-04:00   
1        NaN             NaN      WebClient  2025-09-12T08:04:50-04:00   
2        NaN             NaN      WebClient  2025-09-12T09:41:36-04:00   
5        NaN             NaN      WebClient  2025-09-12T12:31:51-04:00   
6        NaN             NaN      WebClient  2025-09-12T12:32:05-04:00   

           InsertedDate                         ParticipantID  \
0  2025-09-12T12:04:43Z  d1eea705-adbf-4cd0-887b-18e77e8d126c   
1  2025-09-12T12:04:50Z  d1eea705-adbf-4cd0-887b-18e77e8d126c   
2  2025-09-12T13:41:35Z  4640bd77-5394-4d27-b893-b696f108305a   
5  2025-09-12T16:31:52Z  545198c2-150d-400e-8fc2-a58d74cb90a0   
6  2025-09-12T16:32:05Z  545198c2-150d-400e-8fc2-a58d74cb90a0   

  ParticipantIde

In [291]:
# check whether the record of step count start date is the same as the baseline survey date
surveytask['date'] = pd.to_datetime(surveytask['InsertedDate']).dt.date
# survey_task_active = surveytask[surveytask.surveyname == 'MRT - Salience - Message Display']
survey_task_active = surveytask[surveytask.SurveyName == 'MRT - Daily End of day survey and Planning Exercise']
summary_surveytask = (
    survey_task_active
    .groupby('ParticipantIdentifier')['date']
    .agg(date_min='min', date_max='max', n_days='nunique')
    .assign(span_days=lambda df: (pd.to_datetime(df.date_max) -
                                  pd.to_datetime(df.date_min)).dt.days + 1)
)


# make participantidentifier a column name
summary_surveytask.reset_index(inplace=True)
summary_surveytask.rename(columns={'ParticipantIdentifier': 'ParticipantIdentifier'}, inplace=True)



# correct for missing day 0 for participant 22
# summary_surveytask.loc[summary_surveytask['ParticipantIdentifier'] == 22, 'date_min'] = pd.to_datetime('2025-06-29').date()
# summary_surveytask['date_min'] = pd.to_datetime(summary_surveytask['date_min']).dt.normalize()

print(summary_surveytask)

# filter out participants who have span_days longer than 84 days
complete_participant_ids = summary_surveytask[summary_surveytask['span_days'] >= 83]['ParticipantIdentifier'].unique()
print(complete_participant_ids, len(complete_participant_ids))

# remove 112, 138 from complete_participant_ids
# the notification dilvery time is not correct for user 138
# remove 117 because the step count/heart rate data is not correct
# remove 219 because the step count/heart rate data is very sparse
complete_participant_ids = np.setdiff1d(complete_participant_ids, ['112', '138', '117', '219'])
print(complete_participant_ids, len(complete_participant_ids))

# filter out incomplete participants from surveytask and surveyquestionresults
surveytask = surveytask[surveytask['ParticipantIdentifier'].isin(complete_participant_ids)]
surveyquestionresults = surveyquestionresults[surveyquestionresults['ParticipantIdentifier'].isin(complete_participant_ids)]

# print(surveytask.ParticipantIdentifier.unique())
# print(surveyquestionresults.ParticipantIdentifier.unique())

# sort surveytask and surveyquestionresults by participantidentifier and date
surveytask = surveytask.sort_values(by=['ParticipantIdentifier', 'date'])
surveyquestionresults = surveyquestionresults.sort_values(by=['ParticipantIdentifier', 'date'])

# print(surveytask.head())
# print(surveyquestionresults.head())


   ParticipantIdentifier    date_min    date_max  n_days  span_days
0                    106  2025-09-13  2025-11-09      50         58
1                    112  2025-09-22  2025-12-14      78         84
2                    117  2025-11-02  2026-01-25      73         85
3                    118  2025-09-14  2025-12-08      84         86
4                    128  2025-09-12  2025-11-24      62         74
5                    129  2025-09-12  2025-11-16      65         66
6                    138  2025-10-20  2026-01-12      82         85
7                    141  2025-10-20  2026-01-11      84         84
8                    143  2025-09-22  2025-12-14      84         84
9                    151  2025-09-29  2025-12-21      82         84
10                   160  2025-12-01  2026-02-22      84         84
11                   170  2025-10-06  2025-12-28      84         84
12                    18  2025-09-12  2025-09-21      10         10
13                   184  2025-11-02  2026-01-25

## Extract four variables from weekly survey: affective valuation, CAE, perceived helpfulness, perceived pleasantness
- keep participant identifier
- keep delivery date and time
- when the surveys are not responded, we use NA to indicate missingness
- each participant should have exactly 12 rows of four variables

- add adherence indicator and column for participant identifier (j=1 if respond)
- according to the date, the gap should be about 7 days (+/- 1 day)
- use NANs to fill the missing values


- time zone issue: since it's daily 6pm, even though due to the use of UTC, it will be around 10pm. This will not flow to the next day, so it's fine for now!!!!!!!!

In [292]:

def _to_obj(x):
    if isinstance(x, (list, dict)):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        x = x.strip()
        if not x:
            return []
        try:
            return json.loads(x)
        except json.JSONDecodeError:
            return []
    return []

flat_rows = []

for _, row in surveyquestionresults.iterrows():
    pid = row.get("ParticipantIdentifier")
    survey_key = row.get("SurveyKey")
    inserted = row.get("InsertedDate")

    # IMPORTANT: parse SurveyResults here
    survey_results = _to_obj(row.get("StepResults"))
    if isinstance(survey_results, dict):
        survey_results = [survey_results]

    for step in survey_results:
        if not isinstance(step, dict):
            continue

        step_id = step.get("StepIdentifier")
        step_start = step.get("StartDate")
        step_end = step.get("EndDate")

        results = _to_obj(step.get("Results"))
        if isinstance(results, dict):
            results = [results]

        for r in results:
            if not isinstance(r, dict):
                continue

            ans = r.get("Answers")
            if isinstance(ans, list):
                ans_first = ans[0] if ans else np.nan
                ans_raw = "|".join(map(str, ans))
            else:
                ans_first = ans
                ans_raw = str(ans) if ans is not None else np.nan

            flat_rows.append({
                "ParticipantIdentifier": pid,
                "SurveyKey": survey_key,
                "InsertedDate": inserted,
                "StepIdentifier": step_id,
                "StepStartDate": step_start,
                "StepEndDate": step_end,
                "ResultType": r.get("Type"),
                "ResultIdentifier": r.get("ResultIdentifier"),
                "AnswerFirst": ans_first,
                "AnswersRaw": ans_raw,
                "QuestionStartDate": r.get("StartDate"),
                "QuestionEndDate": r.get("EndDate"),
            })

survey_results_flat = pd.DataFrame(flat_rows)
print(survey_results_flat.shape)
survey_results_flat.head()

(10064, 12)


,ParticipantIdentifier,SurveyKey,InsertedDate,StepIdentifier,StepStartDate,StepEndDate,ResultType,ResultIdentifier,AnswerFirst,AnswersRaw,QuestionStartDate,QuestionEndDate
0,118,74f6769e-480e-f011-bb00-0affe8024cad,2025-09-14T22:25:04Z,value,2025-09-14T18:17:47-04:00,2025-09-14T18:19:38-04:00,QuestionResult,value,"Independence, My sons, My health, my dogsdogs","Independence, My sons, My health, my dogsdogs",2025-09-14T18:17:47-04:00,2025-09-14T18:19:38-04:00
1,118,74f6769e-480e-f011-bb00-0affe8024cad,2025-09-14T22:25:04Z,active-lifestyle,2025-09-14T18:19:38-04:00,2025-09-14T18:21:36-04:00,QuestionResult,active-lifestyle,"Maintaining independent lifestyle, traveling t...","Maintaining independent lifestyle, traveling t...",2025-09-14T18:19:38-04:00,2025-09-14T18:21:36-04:00
2,118,74f6769e-480e-f011-bb00-0affe8024cad,2025-09-14T22:25:04Z,carousel-image,2025-09-14T18:21:36-04:00,2025-09-14T18:22:28-04:00,WebViewStepResult,carousel-image,https://orgimages.careevolutionapps.com/d68084...,https://orgimages.careevolutionapps.com/d68084...,2025-09-14T18:21:36-04:00,2025-09-14T18:22:28-04:00
3,118,74f6769e-480e-f011-bb00-0affe8024cad,2025-09-14T22:25:04Z,monthly-goal,2025-09-14T18:22:55-04:00,2025-09-14T18:24:19-04:00,QuestionResult,monthly-goal,Walk the dogs around our loop,Walk the dogs around our loop,2025-09-14T18:22:55-04:00,2025-09-14T18:24:19-04:00
4,118,74f6769e-480e-f011-bb00-0affe8024cad,2025-09-14T22:25:04Z,outro,2025-09-14T18:24:19-04:00,2025-09-14T18:24:54-04:00,WebViewStepResult,Outro,complete,complete,2025-09-14T18:24:19-04:00,2025-09-14T18:24:54-04:00


In [293]:
questions = ["AffectiveValuation", "Exp-tool-1", "Exp-tool-2"] + [f"CAE-{i}" for i in range(1, 13)]

tmp = (
    survey_results_flat
    .loc[survey_results_flat["ResultIdentifier"].isin(questions)]
    .copy()
)

# Keep parsed datetime for ordering/debug (UTC-normalized)
tmp["datetime"] = pd.to_datetime(tmp["QuestionEndDate"], errors="coerce", utc=True)

print(tmp["datetime"].dtype)

# Numeric answer
tmp["value"] = pd.to_numeric(tmp["AnswerFirst"], errors="coerce")

# Local date from original offset timestamp string (YYYY-MM-DD part)
tmp["date"] = pd.to_datetime(
    tmp["QuestionEndDate"].astype(str).str.slice(0, 10),
    errors="coerce"
).dt.date

tmp = tmp.dropna(subset=["datetime", "date"])

df_weekly_survey = (
    tmp.pivot_table(
        index=["ParticipantIdentifier", "date"],
        columns="ResultIdentifier",
        values="value",
        aggfunc="last"
    )
    .reset_index()
)

df_weekly_survey.head()

datetime64[ns, UTC]


ResultIdentifier,ParticipantIdentifier,date,AffectiveValuation,CAE-1,CAE-10,CAE-11,CAE-12,CAE-2,CAE-3,CAE-4,CAE-5,CAE-6,CAE-7,CAE-8,CAE-9,Exp-tool-1,Exp-tool-2
0,118,2025-09-22,5,6,5,5,5,6,5,6,6,5,5,5,4,4,4
1,118,2025-10-06,6,6,5,5,6,5,6,5,5,6,5,5,5,4,4
2,118,2025-10-12,6,6,6,5,5,5,6,6,5,6,5,5,6,2,2
3,118,2025-10-20,5,6,6,5,6,5,5,6,5,6,6,6,6,3,3
4,118,2025-10-27,6,6,6,5,6,7,5,6,6,5,5,5,5,4,4


In [294]:
# extract weekly survey from survey task
survey_task_weekly = surveytask.loc[
    surveytask["SurveyName"].isin([
        "MRT - Weekly Check-in survey",
        "MRT - Monthly check-in survey and goal setting",
        "MRT - Monthly check-in survey (Final month)"
    ])
].copy()

survey_task_weekly.loc[:, "date"] = pd.to_datetime(
    survey_task_weekly["InsertedDate"], errors="coerce"
)

print(survey_task_weekly.head())

             CreatedBy               DueDate          InsertedDate  \
711   Survey Scheduler  2025-09-24T22:05:02Z  2025-09-21T22:05:02Z   
763   Survey Scheduler  2025-09-24T22:05:02Z  2025-09-21T22:05:02Z   
1208  Survey Scheduler  2025-10-01T22:05:04Z  2025-09-28T22:05:04Z   
1331  Survey Scheduler  2025-10-01T22:05:04Z  2025-09-28T22:05:04Z   
1733  Survey Scheduler  2025-10-08T22:05:04Z  2025-10-05T22:05:11Z   

              ModifiedDate                         ParticipantID  \
711   2025-09-21T22:05:02Z  38A26232-F5DE-4F04-8B65-111649661BFE   
763   2025-09-22T11:11:43Z  38A26232-F5DE-4F04-8B65-111649661BFE   
1208  2025-09-28T22:05:04Z  38A26232-F5DE-4F04-8B65-111649661BFE   
1331  2025-10-01T06:00:33Z  38A26232-F5DE-4F04-8B65-111649661BFE   
1733  2025-10-05T22:05:11Z  38A26232-F5DE-4F04-8B65-111649661BFE   

     ParticipantIdentifier      Status                             SurveyKey  \
711                    118  Incomplete  17FB11E8-ADD3-EF11-BAFD-0AFFE8024CAD   
763       

In [295]:
# force both to pandas datetime64[ns] (naive midnight)
df_weekly_survey["date"] = pd.to_datetime(df_weekly_survey["date"], errors="coerce").dt.normalize()
survey_task_weekly["date"] = pd.to_datetime(survey_task_weekly["date"], errors="coerce").dt.normalize()

print(df_weekly_survey["date"].dtype)
print(survey_task_weekly["date"].dtype)

datetime64[ns]
datetime64[ns, UTC]


In [296]:
def fill_weekly_12(df1, df2, id_col="ParticipantIdentifier", date_col="date",
                         tolerance_days=2, weeks=12):
    """
    df1: wide weekly-level survey frame with [id_col, date_col, value columns...]
    df2: wide weekly-level survey task frame with [id_col, date_col, survey task...]
    Returns a frame with exactly `weeks` rows per participant, anchored at the
    participant's earliest observed date, spaced at 7-day intervals (± tolerance).
    """
    # ensure datetime (normalized to date)
    df1 = df1.copy()
    df1[date_col] = pd.to_datetime(df1[date_col], errors="coerce").dt.normalize()

    df2 = df2.copy()
    df2[date_col] = (
        pd.to_datetime(df2[date_col], errors="coerce", utc=True)
        .dt.tz_convert(None)
        .dt.normalize()
    )

    anchor_lookup = (
        df2.groupby(id_col)[date_col]
           .min()  # earliest task date per participant
    )

    # value columns to carry through
    value_cols = [c for c in df1.columns if c not in [id_col, date_col]]

    out_parts = []
    for pid, g in df1.groupby(id_col, sort=False):

        anchor = anchor_lookup.get(pid)  # earliest observed date

        # expected weekly slots
        slots = pd.DataFrame({
            "idx": np.arange(weeks, dtype=int),
            id_col: pid,
            "date": [anchor + pd.Timedelta(days=7*i) for i in range(weeks)]
        })

        # assign each observed date to the nearest weekly slot
        g = g.assign(
            idx=np.round((g[date_col] - anchor).dt.days / 7.0).astype(int)
        )
        # distance from its slot in absolute days
        g = g.assign(
            _slot_date=lambda d: anchor + pd.to_timedelta(d["idx"]*7, unit="D"),
            _abs_diff=lambda d: (d[date_col] - d["_slot_date"]).abs()
        )
        # keep only rows that fall within tolerance and within slot range
        g = g[(g["idx"] >= 0) & (g["idx"] < weeks) &
                (g["_abs_diff"] <= pd.Timedelta(days=tolerance_days))]

        # if multiple observed dates map to the same slot, keep the nearest
        g = (g.sort_values(["_abs_diff", date_col])
                .drop_duplicates(subset=["idx"], keep="first"))

        # prepare for merge
        keep_cols = [id_col, "idx", date_col] + value_cols
        g = g[keep_cols].rename(columns={date_col: "observed_date"})

        # merge slots with matched observations
        merged = slots.merge(g, on=[id_col, "idx"], how="left")

        # present indicator: 1 if we matched a row to this slot, else 0
        merged["week_present"] = merged["observed_date"].notna().astype(int)

        # ensure all value cols exist (NaN if missing)
        for c in value_cols:
            if c not in merged.columns:
                merged[c] = np.nan

        # tidy columns
        merged = merged.drop(columns=["idx"]).sort_values(["date"])
        out_parts.append(merged)

    out = pd.concat(out_parts, ignore_index=True)

    # order columns: id, date (expected slot), observed_date, present, then values
    ordered = [id_col, "date", "observed_date", "week_present"] + value_cols
    out = out[ordered]

    return out


df_weekly_filled = fill_weekly_12(df_weekly_survey, survey_task_weekly)

# add week numbers as well:
df_weekly_filled["week"] = (
    df_weekly_filled.groupby("ParticipantIdentifier").cumcount() + 1
)

# print(df_weekly_filled)

# print the number of adherence=1 for each participant
all_counts = (
    df_weekly_filled.groupby("ParticipantIdentifier")["week_present"]
    .sum()
    .astype(int)
)
print(all_counts)
# print the number of survey results for each participant in the original dataframe
print(df_weekly_survey.groupby("ParticipantIdentifier").size())

# # check what happens to participant 33
# print(df_weekly_survey[df_weekly_survey.ParticipantIdentifier == "117"])

# # check what happens to participant 33 in the filled dataframe
# print(df_weekly_filled[df_weekly_filled.ParticipantIdentifier == "117"])

# participant 33 completed a survey on 2025-10-09 which is not within the 12 weeks

# save the filled dataframe
df_weekly_filled.to_csv(folder / "df_weekly_filled.csv", index=False)

ParticipantIdentifier
118    10
141     6
143    10
151     2
160     6
170     9
184     3
188    11
195    12
204    12
225    12
Name: week_present, dtype: int64
ParticipantIdentifier
118    10
141     6
143    10
151     3
160     7
170    10
184     4
188    11
195    12
204    12
225    12
dtype: int64


## Extract variables from end-of-day survey: affective reflection, anticipated affect

- keep participant identifier
- keep delivery date and time
- when the surveys are not responded, we use NA to indicate missingness
- each participant should have about 84 rows of 2 variables
- rows after 84 are removed..
- **Important**: the start date should be decided according to survey task not survey question results, as the latter only has data when the user answers the survey!!!

- add missingness indicator and column for participant identifier
- according to the date, the gap should be about 84 days (+/- 1 day)
- use NANs to fill the missing values
- missingness indicator: 1 for present, 0 for missing

- Time zone issue: it's bedtime minus 2 or 4, so sometimes it can flow to next day.........
- Let's do a minus 4 for all users 
- ignore winter time or other time zones in addition to eastern time.

In [297]:
questions = ["AffectiveReflection", "AnticipatedAffect"]

tmp = (
    survey_results_flat
    .loc[survey_results_flat["ResultIdentifier"].isin(questions)]
    .copy()
)

# Same pattern as weekly block
tmp["datetime"] = pd.to_datetime(tmp["QuestionEndDate"], errors="coerce", utc=True)
tmp["value"] = pd.to_numeric(tmp["AnswerFirst"], errors="coerce")
tmp["date"] = pd.to_datetime(
    tmp["QuestionEndDate"].astype(str).str.slice(0, 10),
    errors="coerce"
).dt.date

tmp = tmp.dropna(subset=["datetime", "date"])

df_daily_survey = (
    tmp.pivot_table(
        index=["ParticipantIdentifier", "date"],
        columns="ResultIdentifier",
        values="value",
        aggfunc="last"
    )
    .reset_index()
)

rename_map = {
    "AffectiveReflection": "affective_reflection",
    "AnticipatedAffect": "anticipated_affect",
}
df_daily_survey = df_daily_survey.rename(columns=rename_map)

desired_cols = ["ParticipantIdentifier", "date", "affective_reflection", "anticipated_affect"]
for c in desired_cols:
    if c not in df_daily_survey.columns:
        df_daily_survey[c] = np.nan

df_daily_survey = df_daily_survey[desired_cols].sort_values(["ParticipantIdentifier", "date"])
print(df_daily_survey.head())

ResultIdentifier ParticipantIdentifier        date  affective_reflection  \
0                                  118  2025-09-14                   5.0   
1                                  118  2025-09-16                   6.0   
2                                  118  2025-09-17                   5.0   
3                                  118  2025-09-22                   6.0   
4                                  118  2025-09-24                   6.0   

ResultIdentifier  anticipated_affect  
0                                5.0  
1                                6.0  
2                                6.0  
3                                6.0  
4                                6.0  


In [298]:
# extract daily survey from survey task
survey_task_eod = surveytask.loc[surveytask.SurveyName == 'MRT - Daily End of day survey and Planning Exercise'].copy()
survey_task_eod['date'] = pd.to_datetime(survey_task_eod['InsertedDate'])

print(survey_task_eod)

      CreatedBy               DueDate          InsertedDate  \
214         NaN  2025-09-15T23:02:49Z  2025-09-14T23:02:49Z   
332         NaN  2025-09-16T23:06:06Z  2025-09-15T23:06:06Z   
379         NaN  2025-09-17T23:04:18Z  2025-09-16T23:04:18Z   
458         NaN  2025-09-18T23:04:42Z  2025-09-17T23:04:42Z   
547         NaN  2025-09-19T23:04:11Z  2025-09-18T23:04:12Z   
...         ...                   ...                   ...   
13279       NaN  2026-03-06T23:01:25Z  2026-03-05T23:01:25Z   
13339       NaN  2026-03-07T23:01:10Z  2026-03-06T23:01:11Z   
13369       NaN  2026-03-07T23:01:10Z  2026-03-06T23:01:11Z   
13403       NaN  2026-03-08T23:01:18Z  2026-03-07T23:01:18Z   
13449       NaN  2026-03-09T22:01:07Z  2026-03-08T22:01:07Z   

               ModifiedDate                         ParticipantID  \
214    2025-09-14T23:05:58Z  38A26232-F5DE-4F04-8B65-111649661BFE   
332    2025-09-16T07:01:19Z  38A26232-F5DE-4F04-8B65-111649661BFE   
379    2025-09-17T00:55:12Z  38A2623

In [299]:
def fill_daily_85(df1, id_col="ParticipantIdentifier", date_col="date", days=85):
    """
    df1: wide daily-level survey question results frame with [id_col, date_col, value columns...]
    df2: wide daily-level survey task frame with [id_col, date_col, survey task...]
    Returns a frame with exactly `days` rows per participant, anchored at the
    participant's earliest observed date, spaced at 1-day intervals (± tolerance).
    """
    # ensure datetime (normalized to date)
    df1 = df1.copy()
    df1[date_col] = pd.to_datetime(df1[date_col]).dt.normalize()

    # df2 = df2.copy()
    # df2[date_col] = pd.to_datetime(df2[date_col]).dt.normalize()

    # anchor_lookup = (
    #     df2.groupby(id_col)[date_col]
    #        .min()  # earliest task date per participant
    # )

    # value columns to carry through
    value_cols = [c for c in df1.columns if c not in [id_col, date_col]]

    out_parts = []
    for pid, g in df1.groupby(id_col, sort=False):

        # anchor = anchor_lookup.get(pid)  # earliest observed date
        anchor = summary_surveytask.loc[summary_surveytask['ParticipantIdentifier'] == pid, 'date_min'].iloc[0]
        anchor = pd.to_datetime(anchor).normalize()
        
        # expected daily slots
        slots = pd.DataFrame({
            "idx": np.arange(days, dtype=int),
            id_col: pid,
            "date": [anchor + pd.Timedelta(days=1*i) for i in range(days)]
        })

        # assign each observed date to the nearest daily slot
        g = g.assign(
            idx=np.round((g[date_col] - anchor).dt.days / 1.0).astype(int)
        )
        # distance from its slot in absolute days
        g = g.assign(
            _slot_date=lambda d: anchor + pd.to_timedelta(d["idx"]*1, unit="D"),
            _abs_diff=lambda d: (d[date_col] - d["_slot_date"]).abs()
        )
        # keep only rows that fall within tolerance and within slot range
        g = g[(g["idx"] >= 0) & (g["idx"] < days)]

        # if multiple observed dates map to the same slot, keep the nearest
        g = (g.sort_values(["_abs_diff", date_col])
                .drop_duplicates(subset=["idx"], keep="first"))

        # prepare for merge
        keep_cols = [id_col, "idx", date_col] + value_cols
        g = g[keep_cols].rename(columns={date_col: "observed_date"})

        # merge slots with matched observations
        merged = slots.merge(g, on=[id_col, "idx"], how="left")

        # present indicator: 1 if we matched a row to this slot, else 0
        merged["daily_present"] = merged["observed_date"].notna().astype(int)

        # ensure all value cols exist (NaN if missing)
        for c in value_cols:
            if c not in merged.columns:
                merged[c] = np.nan

        # tidy columns
        merged = merged.drop(columns=["idx"]).sort_values(["date"])
        out_parts.append(merged)

    out = pd.concat(out_parts, ignore_index=True)

    # order columns: id, date (expected slot), observed_date, present, then values
    ordered = [id_col, "date", "observed_date", "daily_present"] + value_cols
    out = out[ordered]

    return out


df_daily_filled = fill_daily_85(df_daily_survey)
# , survey_task_eod)

# add day numbers as well:
df_daily_filled["day"] = (
    df_daily_filled.groupby("ParticipantIdentifier").cumcount() + 1
)



# print the number of missingness=1 for each participant
print(df_daily_filled[df_daily_filled.daily_present == 1].groupby("ParticipantIdentifier").size())

# print the number of survey results for each participant in the original dataframe
print(df_daily_survey.groupby("ParticipantIdentifier").size())

# check what happens to participant 33
# print(df_daily_survey[df_daily_survey.ParticipantIdentifier == 75])

# check what happens to participant 33 in the filled dataframe
# print(df_daily_filled[df_daily_filled.participantidentifier == 75])

df_daily_filled.to_csv(folder / "df_daily_filled.csv", index=False)


ParticipantIdentifier
118    44
141    22
143    64
151     3
160    18
170    41
184     8
188    60
195    34
204    83
225    79
dtype: int64
ParticipantIdentifier
118    44
141    22
143    64
151     3
160    18
170    41
184     8
188    60
195    35
204    83
225    79
dtype: int64


## Extract wakeup and bedtime for each participant
- this is useful for filling in the walking suggestions delivery time!

In [300]:
# get baseline surveykey
# survey_key_baseline = surveytask[surveytask.surveyname == 'MRT - Personalize HeartSteps'].surveykey.values[0]

# get wakeup and bedtime for each user
# key fields: resultidentifier: Wakeday Wakeup, Wakeday Bedtime, Weekend Wakeup, Weekend Bedtime
from typing import Any


weekday_wakeup_all = []
weekday_bedtime_all = []
weekend_wakeup_all = []
weekend_bedtime_all = []

question_enddate_all = []

for participant_id in complete_participant_ids:
    # print(participant_id)
    subset = survey_results_flat[survey_results_flat.ParticipantIdentifier == participant_id]
    wake_weekday = (
        subset[subset.ResultIdentifier == 'Weekday Wakeup']['AnswersRaw']
        .str.strip('[]')
        .pipe(pd.to_datetime, format='%I:%M %p', errors='coerce')
    )
    weekday_wakeup_all.append(wake_weekday.dt.time.to_list())

    question_enddate = pd.to_datetime(
        subset.loc[subset["ResultIdentifier"] == "Weekday Wakeup", "QuestionEndDate"],
        errors="coerce",
        utc=True
    )
    question_enddate_all.append(question_enddate.dt.date.to_list())


    bed_weekday = (
        subset[subset.ResultIdentifier == 'Weekday Bedtime']['AnswersRaw']
        .str.strip('[]')
        .pipe(pd.to_datetime, format='%I:%M %p', errors='coerce')
    )
    weekday_bedtime_all.append(bed_weekday.dt.time.to_list())

    wake_weekend = (
        subset[subset.ResultIdentifier == 'Weekend Wakeup']['AnswersRaw']
        .str.strip('[]')
        .pipe(pd.to_datetime, format='%I:%M %p', errors='coerce')
    )
    weekend_wakeup_all.append(wake_weekend.dt.time.to_list())

    bed_weekend = (
        subset[subset.ResultIdentifier == 'Weekend Bedtime']['AnswersRaw']
        .str.strip('[]')
        .pipe(pd.to_datetime, format='%I:%M %p', errors='coerce')
    )
    weekend_bedtime_all.append(bed_weekend.dt.time.to_list())       

from itertools import zip_longest

rows = []
for pid, time1s, time2s, time3s, time4s, dates in zip(
    complete_participant_ids,
    weekday_wakeup_all,
    weekday_bedtime_all,
    weekend_wakeup_all,
    weekend_bedtime_all,
    question_enddate_all
):
    for time1, time2, time3, time4, date in zip_longest(
        time1s, time2s, time3s, time4s, dates
    ):
        rows.append({
            "ParticipantIdentifier": pid,
            "WeekdayWakeup": time1,
            "WeekdayBedtime": time2,
            "WeekendWakeup": time3,
            "WeekendBedtime": time4,
            "QuestionEndDate": date,
        })

df_wakeup_bedtime = pd.DataFrame(rows)

# error with participant 22, 37, 13
# mask_22 = df_wakeup_bedtime['ParticipantIdentifier'] == 22
# df_wakeup_bedtime.loc[mask_22, 'WeekdayBedtime'] = pd.to_datetime('23:00:00').time()
# df_wakeup_bedtime.loc[mask_22, 'WeekendBedtime'] = pd.to_datetime('23:59:59').time()  # '24:00:00' is invalid
# df_wakeup_bedtime.loc[df_wakeup_bedtime['ParticipantIdentifier'] == 37, 'WeekendWakeup'] = pd.to_datetime('07:00:00').time()
# df_wakeup_bedtime.loc[df_wakeup_bedtime['ParticipantIdentifier'] == 13, 'WeekendBedtime'] = pd.to_datetime('23:59:59').time()


# recover wakeup and bedtime for some users who have missing data
participant_without_wakeup_bedtime = complete_participant_ids[
    ~np.isin(complete_participant_ids, df_wakeup_bedtime["ParticipantIdentifier"].unique())
]
print(participant_without_wakeup_bedtime)

# Manual defaults for participants missing from survey extraction
MANUAL_WAKEUP_BEDTIME = {
    "117": {
        "WeekdayWakeup": "07:00:00",
        "WeekendWakeup": "08:30:00",
        "WeekdayBedtime": "23:00:00",
        "WeekendBedtime": "23:00:00",
    },
    "118": {
        "WeekdayWakeup": "07:30:00",
        "WeekendWakeup": "07:30:00",
        "WeekdayBedtime": "23:00:00",
        "WeekendBedtime": "23:00:00",
    },
    "143": {
        "WeekdayWakeup": "05:30:00",
        "WeekendWakeup": "06:30:00",
        "WeekdayBedtime": "23:00:00",
        "WeekendBedtime": "23:00:00",
    },
    "151": {
        "WeekdayWakeup": "05:30:00",
        "WeekendWakeup": "06:30:00",
        "WeekdayBedtime": "23:00:00",
        "WeekendBedtime": "23:00:00",
    },
}
def _to_time(s: str):
    return pd.to_datetime(s).time()
# --- apply / append ---
for pid, cols in MANUAL_WAKEUP_BEDTIME.items():
    mask = df_wakeup_bedtime["ParticipantIdentifier"] == pid
    if mask.any():
        # update existing row(s) — use first match if duplicates
        idx = df_wakeup_bedtime.index[mask][0]
        for k, v in cols.items():
            df_wakeup_bedtime.loc[idx, k] = _to_time(v)
        if "QuestionEndDate" in df_wakeup_bedtime.columns and pd.isna(df_wakeup_bedtime.loc[idx, "QuestionEndDate"]):
            df_wakeup_bedtime.loc[idx, "QuestionEndDate"] = pd.NaT
    else:
        row = {"ParticipantIdentifier": pid, "QuestionEndDate": pd.NaT}
        for k, v in cols.items():
            row[k] = _to_time(v)
        df_wakeup_bedtime = pd.concat(
            [df_wakeup_bedtime, pd.DataFrame([row])],
            ignore_index=True,
        )
print(df_wakeup_bedtime)


['118' '143' '151']
   ParticipantIdentifier WeekdayWakeup WeekdayBedtime WeekendWakeup  \
0                    141      06:00:00       22:00:00      06:00:00   
1                    160      07:00:00       22:00:00      07:00:00   
2                    170      06:00:00       22:00:00      06:00:00   
3                    184      09:30:00       22:30:00      09:30:00   
4                    188      07:00:00       22:00:00      08:00:00   
5                    188      20:00:00       23:00:00      08:00:00   
6                    188      08:00:00       23:00:00      08:00:00   
7                    195      07:00:00       22:00:00      08:00:00   
8                    204      08:00:00       23:00:00      08:00:00   
9                    204      08:00:00       23:00:00      08:00:00   
10                   225      06:30:00       22:00:00      06:30:00   
11                   225      06:30:00       22:00:00      06:30:00   
12                   117      07:00:00       23:00:00    

## Extract daily engagement(number of view viewed) data

- for engagement, we may have yesterday's engagement data for day 1
- this is different from survey variables...

In [301]:
# Select date range based on daily survey start and end date
pageview = pd.read_csv(folder / 'AnalyticsEvents_ViewViewed.csv')
pageview = convert_utc_columns_to_user_local(
    pageview,
    datetime_cols=["Timestamp"],
    participant_col="ParticipantIdentifier",
    join_date_col="Timestamp",
)
pageview_selected = []
for participant_id in complete_participant_ids:
    pageview_participant = pageview[pageview['ParticipantIdentifier'] == participant_id].copy()
    pageview_participant['Date'] = pageview_participant['Timestamp'].dt.date
    summary_row = summary_surveytask.loc[
        summary_surveytask['ParticipantIdentifier'] == participant_id
    ].iloc[0]  # assumes one row per participant

   
    start_date = pd.to_datetime(summary_row['date_min']).date() - pd.Timedelta(days=1)
    end_date = start_date + pd.Timedelta(days=85)
    pageview_participant = pageview_participant[pageview_participant.Date >= start_date]
    pageview_participant = pageview_participant[pageview_participant.Date <= end_date]
    pageview_selected.append(pageview_participant)

pageview_selected = pd.concat(pageview_selected)
pageview_selected = pageview_selected[["ParticipantIdentifier", "Timestamp"]]
# print(pageview_selected[pageview_selected['ParticipantIdentifier'] == 37])

In [302]:

daily_pageview_list = []
for participant_id in complete_participant_ids:
    pageview_participant = pageview_selected[(pageview_selected['ParticipantIdentifier'] == participant_id)].copy()
    df_wakeup_bedtime_participant = df_wakeup_bedtime.loc[df_wakeup_bedtime['ParticipantIdentifier'] == participant_id].copy()

    min_date = summary_surveytask.loc[
        summary_surveytask['ParticipantIdentifier'] == participant_id
    ].iloc[0].date_min - pd.Timedelta(days=7)
    date_range_length = 92

    for i in range(date_range_length):
        date = min_date + pd.Timedelta(days=i)
        # print(date)
        if df_wakeup_bedtime_participant.shape[0] == 1:
            weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[0]
            weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[0]
            weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[0]
            weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[0]
        else:
            change_date = df_wakeup_bedtime_participant['QuestionEndDate']
            weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[0]
            weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[0]
            weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[0]
            weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[0]
            
            for j in range(len(df_wakeup_bedtime_participant)):
                change_date = df_wakeup_bedtime_participant['QuestionEndDate'].iloc[j]
                
                # Check if this is the applicable period
                if j < len(df_wakeup_bedtime_participant) - 1:
                    next_change_date = df_wakeup_bedtime_participant['QuestionEndDate'].iloc[j + 1]
                    if change_date <= date < next_change_date:
                        weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[j]
                        weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[j]
                        weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[j]
                        weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[j]
                        break
                else:
                    # Last entry - applies from change_date onwards
                    if change_date <= date:
                        weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[j]
                        weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[j]
                        weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[j]
                        weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[j]
                        break
        
        # Determine wakeup time based on weekday/weekend
        is_weekday = date.weekday() < 5
        wakeup_time = weekday_wakeup if is_weekday else weekend_wakeup
        bedtime_time = weekday_bedtime if is_weekday else weekend_bedtime


        # filter out the days with less than 8 hours of wearing fitbit (more than 8 hours of heart rate =0 or nan
        # within the wakeup and bedtime)
        pageview_participant_date = pageview_participant[pageview_participant.Timestamp.dt.date == date]
        pageview_participant_date = pageview_participant_date[pageview_participant_date.Timestamp.dt.time >= wakeup_time]
        pageview_participant_date = pageview_participant_date[pageview_participant_date.Timestamp.dt.time <= bedtime_time]
        

        # Calculate time span from first to last valid reading
        if pageview_participant_date.shape[0] > 0:
            valid_count = pageview_participant_date.shape[0] 
        else:
            valid_count = 0
        # Mark as missing if less than 8 hours of valid data

        daily_pageview_list.append({
            'ParticipantIdentifier': participant_id,
            'Date': pd.to_datetime(date),
            'DailyPageviewCount': valid_count
        })


df_daily_pageview = pd.DataFrame(daily_pageview_list)

# get yesterday's pageview count
df_daily_pageview['YesterdayPageviewCount'] = (
    df_daily_pageview
    .sort_values(['ParticipantIdentifier', 'Date'])
    .groupby('ParticipantIdentifier')['DailyPageviewCount']
    .shift(1)
    .fillna(0)            # or np.nan, or dropna() later
    .astype(int)
)

# get past 7 days pageview exponential moving average
df_daily_pageview['Past7DaysPageviewEMA'] = (
    df_daily_pageview['DailyPageviewCount'].ewm(span=7, adjust=False).mean()
)

# get tomorrow's pageview count
df_daily_pageview['TomorrowPageviewCount'] = (
    df_daily_pageview['DailyPageviewCount'].shift(-1)
)


df_daily_pageview.to_csv(os.path.join(folder, 'df_daily_pageview.csv'), index=False)
# print(df_daily_pageview[df_daily_pageview['participantidentifier'] == 31])



# remove the last row for each participant
df_daily_pageview = df_daily_pageview.groupby('ParticipantIdentifier').apply(lambda x: x.iloc[:-1])

print(df_daily_pageview)


                           ParticipantIdentifier       Date  \
ParticipantIdentifier                                         
118                   0                      118 2025-09-07   
                      1                      118 2025-09-08   
                      2                      118 2025-09-09   
                      3                      118 2025-09-10   
                      4                      118 2025-09-11   
...                                          ...        ...   
225                   1006                   225 2026-03-04   
                      1007                   225 2026-03-05   
                      1008                   225 2026-03-06   
                      1009                   225 2026-03-07   
                      1010                   225 2026-03-08   

                            DailyPageviewCount  YesterdayPageviewCount  \
ParticipantIdentifier                                                    
118                   0         

/var/folders/hz/q4hdsnpj1h50x_jpx8v5mrvc0000gn/T/ipykernel_83343/270276863.py:103: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_daily_pageview = df_daily_pageview.groupby('ParticipantIdentifier').apply(lambda x: x.iloc[:-1])


In [303]:


hourly_pageview_list = []

for participant_id in complete_participant_ids:
    pageview_participant = pageview_selected[pageview_selected['ParticipantIdentifier'] == participant_id].copy()
    df_wakeup_bedtime_participant = df_wakeup_bedtime.loc[df_wakeup_bedtime['ParticipantIdentifier'] == participant_id].copy()

    # we minus 1 day because we want to include yesterday's data of day 1
    # TODO: figured out we may not need to minus 1 day because we start modeling step counts after the first day of end of day survey
    min_date = summary_surveytask.loc[
        summary_surveytask['ParticipantIdentifier'] == participant_id
    ].iloc[0].date_min 
    date_range_length = 85

    for i in range(date_range_length):
        date = min_date + pd.Timedelta(days=i)
        
        if df_wakeup_bedtime_participant.shape[0] == 1:
            weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[0]
            weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[0]
            weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[0]
            weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[0]
        else:
            change_date = df_wakeup_bedtime_participant['QuestionEndDate']
            weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[0]
            weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[0]
            weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[0]
            weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[0]
            
            for j in range(len(df_wakeup_bedtime_participant)):
                change_date = df_wakeup_bedtime_participant['QuestionEndDate'].iloc[j]
                
                # Check if this is the applicable period
                if j < len(df_wakeup_bedtime_participant) - 1:
                    next_change_date = df_wakeup_bedtime_participant['QuestionEndDate'].iloc[j + 1]
                    if change_date <= date < next_change_date:
                        weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[j]
                        weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[j]
                        weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[j]
                        weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[j]
                        break
                else:
                    # Last entry - applies from change_date onwards
                    if change_date <= date:
                        weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[j]
                        weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[j]
                        weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[j]
                        weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[j]
                        break
        
        # Determine wakeup time based on weekday/weekend
        is_weekday = date.weekday() < 5
        wakeup_time = weekday_wakeup if is_weekday else weekend_wakeup
        bedtime_time = weekday_bedtime if is_weekday else weekend_bedtime

        # Get all heart rate data for this date
        pageview_participant_date = pageview_participant[pageview_participant.Timestamp.dt.date == date]
        pageview_participant_date = pageview_participant_date[pageview_participant_date.Timestamp.dt.time >= wakeup_time]
        pageview_participant_date = pageview_participant_date[pageview_participant_date.Timestamp.dt.time <= bedtime_time]
        
        # Create hourly bins from wakeup to bedtime
        wakeup_datetime = pd.Timestamp.combine(date, wakeup_time)
        bedtime_datetime = pd.Timestamp.combine(date, bedtime_time)
        
        # Generate hourly time bins
        num_decisions = 2 # 2 decision points per day wakeup + 1, wakeup + 6
        current_decision = 0
        while current_decision < num_decisions:
            start_window = wakeup_datetime + pd.Timedelta(hours=1) if current_decision == 0 else wakeup_datetime + pd.Timedelta(hours=6)
            end_window = start_window + pd.Timedelta(hours=4)
            
            # Filter data for this hour
            view_data = pageview_participant_date[
                (pageview_participant_date['Timestamp'] >= start_window) &
                (pageview_participant_date['Timestamp'] < end_window)
            ]
            

            
            # Calculate time span of valid data in this hour
            if view_data.shape[0] > 0:
                valid_count = view_data.shape[0]
            else:
                valid_count = 0

            
            hourly_pageview_list.append({
                'ParticipantIdentifier': participant_id,
                'Date': date,
                'DecisionTime': current_decision,
                'DatetimeStart': start_window,
                'HourlyPageviewCount': valid_count
            })
            
            current_decision += 1

df_hourly_pageview = pd.DataFrame(hourly_pageview_list)
print(df_hourly_pageview)

# save the dataframe
df_hourly_pageview.to_csv(folder / 'hourly_pageview.csv', index=False)


     ParticipantIdentifier        Date  DecisionTime       DatetimeStart  \
0                      118  2025-09-14             0 2025-09-14 08:30:00   
1                      118  2025-09-14             1 2025-09-14 13:30:00   
2                      118  2025-09-15             0 2025-09-15 08:30:00   
3                      118  2025-09-15             1 2025-09-15 13:30:00   
4                      118  2025-09-16             0 2025-09-16 08:30:00   
...                    ...         ...           ...                 ...   
1865                   225  2026-03-07             1 2026-03-07 12:30:00   
1866                   225  2026-03-08             0 2026-03-08 07:30:00   
1867                   225  2026-03-08             1 2026-03-08 12:30:00   
1868                   225  2026-03-09             0 2026-03-09 07:30:00   
1869                   225  2026-03-09             1 2026-03-09 12:30:00   

      HourlyPageviewCount  
0                       0  
1                       0  
2  

## Extract all actions data and click data at the same time
- twice daily walking suggestions map to hourly, twice-daily, daily, and weekly
- daily planning prompts map to hourly, twice-daily, daily, and weekly
- daily salience message map to hourly, twice-daily, daily, and weekly


- Time zone: timestamp minus 4 hours for now!!!

## Notes on click data
- for walking suggestions/salience messages, an important variable is whether the user opens the notification
- for planning prompts, an important variable is whether the user fills in the survey!

- maybe the users saw the notifictaion but didn't open it... so perhaps we can also ignore this for now!!!!!!!

### Missing data problem:
- for user 13, end-of-day survey + planning action delivery information is missing on day 7/26 and 8/3. This is weired. 
- TODO: check project device data, which is more accurate than notification sent csv...

In [304]:
# load analytics_event_push data
push_sent = pd.read_csv(folder / 'AnalyticsEvents_PushNotificationSent.csv')

# to local time
push_sent = convert_utc_columns_to_user_local(
    push_sent,
    datetime_cols=["Timestamp"],
    participant_col="ParticipantIdentifier",
    join_date_col="Timestamp",
)


push_open = pd.read_csv(folder / 'AnalyticsEvents_PushNotificationOpened.csv')

# to local time
push_open = convert_utc_columns_to_user_local(
    push_open,
    datetime_cols=["Timestamp"],
    participant_col="ParticipantIdentifier",
    join_date_col="Timestamp",
)


In [305]:
gif_rows      = push_sent.loc[push_sent['Properties.NotificationIdentifier'].str.startswith('gif', na=False)].copy()
end_rows      = push_sent.loc[push_sent['Properties.NotificationIdentifier'].str.startswith('endOfDay', na=False)].copy()
salience_rows = push_sent.loc[push_sent['Properties.NotificationIdentifier'].str.startswith('salience', na=False)].copy()

# delete repeated rows
gif_rows = gif_rows.drop_duplicates()
end_rows = end_rows.drop_duplicates()
salience_rows = salience_rows.drop_duplicates()

# print(end_rows[end_rows['ParticipantIdentifier'] == "117"])
# print(end_rows.head())
# print(salience_rows.head())


gif_rows_open      = push_open.loc[push_open['Properties.NotificationIdentifier'].str.startswith('gif', na=False)].copy()
# end_rows_open      = push_open.loc[push_open['notification_id'].str.startswith('endOfDay', na=False)].copy()
salience_rows_open = push_open.loc[push_open['Properties.NotificationIdentifier'].str.startswith('salience', na=False)].copy()

# print(gif_rows_open[gif_rows_open['participantidentifier'] == 75])
# print(salience_rows_open.head())


In [313]:
# check the date range of the end of day survey for each participant
min_date_list = []
max_date_list = []
date_range_length_list = []
# complete_participant_ids = [complete_participant_ids, "251"]
for participant_id in complete_participant_ids:
    end_rows_participant = end_rows.loc[end_rows['ParticipantIdentifier'] == participant_id].copy()
    end_rows_participant['Timestamp'] = pd.to_datetime(end_rows_participant['Timestamp'])

    min_date = end_rows_participant['Timestamp'].min()
    max_date = end_rows_participant['Timestamp'].max()

    print(f"Participant {participant_id} end of day survey date range: {min_date} to {max_date}")
    min_date_list.append(min_date)
    max_date_list.append(max_date)
    # get the length of the date range
    date_range_length = (max_date - min_date).days + 1
    date_range_length_list.append(date_range_length)

print(min_date_list)
print(max_date_list)
print(date_range_length_list)

end_rows_251 = end_rows.loc[end_rows['ParticipantIdentifier'] == "251"].copy()
end_rows_251['Timestamp'] = pd.to_datetime(end_rows_251['Timestamp'])
min_date_251 = end_rows_251['Timestamp'].min()
max_date_251 = end_rows_251['Timestamp'].max()
print(f"Participant 251 end of day survey date range: {min_date_251} to {max_date_251}")
print(end_rows_251)



Participant 118 end of day survey date range: 2025-09-14 19:00:11 to 2025-12-07 19:00:52
Participant 141 end of day survey date range: 2025-10-20 18:00:09 to 2026-01-11 18:00:40
Participant 143 end of day survey date range: 2025-09-22 17:00:31 to 2025-12-14 17:02:43
Participant 151 end of day survey date range: 2025-09-29 05:30:07 to 2025-12-21 05:30:29
Participant 160 end of day survey date range: 2025-12-01 18:00:11 to 2026-02-22 23:00:52
Participant 170 end of day survey date range: 2025-10-06 18:00:13 to 2025-12-28 18:00:40
Participant 184 end of day survey date range: 2025-11-02 18:30:29 to 2026-01-25 18:32:56
Participant 188 end of day survey date range: 2025-12-01 18:00:13 to 2026-02-23 00:00:35
Participant 195 end of day survey date range: 2025-12-08 18:00:05 to 2026-03-01 23:00:35
Participant 204 end of day survey date range: 2025-11-16 19:00:07 to 2026-02-09 00:02:41
Participant 225 end of day survey date range: 2025-12-15 18:00:29 to 2026-03-08 22:01:04
[Timestamp('2025-09-1

In [312]:
# print walkings suggestions range
# check the date range of the end of day survey for each participant
min_date_list_gif = []
max_date_list_gif = []
date_range_length_list_gif = []
for participant_id in complete_participant_ids:
    gif_rows_participant = gif_rows.loc[gif_rows['ParticipantIdentifier'] == participant_id].copy()
    gif_rows_participant['Timestamp'] = pd.to_datetime(gif_rows_participant['Timestamp'])

    min_date = gif_rows_participant['Timestamp'].min()
    max_date = gif_rows_participant['Timestamp'].max()

    # print(f"Participant {participant_id} gif date range: {min_date} to {max_date}")
    min_date_list_gif.append(min_date)
    max_date_list_gif.append(max_date)

    # get the length of the date range
    date_range_length_gif = (max_date - min_date).days + 1
    date_range_length_list_gif.append(date_range_length_gif)

# print(min_date_list_gif)
# print(max_date_list_gif)
# print(date_range_length_list_gif)

gif_rows_251 = gif_rows.loc[gif_rows['ParticipantIdentifier'] == "251"].copy()
gif_rows_251['Timestamp'] = pd.to_datetime(gif_rows_251['Timestamp'])
min_date_251 = gif_rows_251['Timestamp'].min()
max_date_251 = gif_rows_251['Timestamp'].max()
print(f"Participant 251 gif date range: {min_date_251} to {max_date_251}")
print(gif_rows_251)


# print(gif_rows_251.head())

# print(gif_rows_251.tail())



Participant 251 gif date range: 2025-12-29 09:00:11 to 2026-02-13 19:02:47
                              ParticipantID ParticipantIdentifier  \
10609  69b72f3f-07e6-44c9-8951-0be94043a501                   251   
10666  69b72f3f-07e6-44c9-8951-0be94043a501                   251   
10723  69b72f3f-07e6-44c9-8951-0be94043a501                   251   
10765  69b72f3f-07e6-44c9-8951-0be94043a501                   251   
10802  69b72f3f-07e6-44c9-8951-0be94043a501                   251   
10901  69b72f3f-07e6-44c9-8951-0be94043a501                   251   
10972  69b72f3f-07e6-44c9-8951-0be94043a501                   251   
11056  69b72f3f-07e6-44c9-8951-0be94043a501                   251   
11091  69b72f3f-07e6-44c9-8951-0be94043a501                   251   
11137  69b72f3f-07e6-44c9-8951-0be94043a501                   251   
11191  69b72f3f-07e6-44c9-8951-0be94043a501                   251   
11283  69b72f3f-07e6-44c9-8951-0be94043a501                   251   
11550  69b72f3f-07e6-44c9-89

In [241]:
end_rows['planning_prompt'] = 0
end_rows.loc[end_rows['Properties.NotificationIdentifier'].str.startswith('endOfDay_Planning', na=False), 'planning_prompt'] = 1
# print(end_rows.head())

end_rows['Timestamp'] = pd.to_datetime(end_rows['Timestamp'])
end_rows['date'] = end_rows['Timestamp'].dt.date
end_rows['time'] = end_rows['Timestamp'].dt.time
df_end_all = end_rows[['ParticipantIdentifier', 'date', 'time', 'planning_prompt']]


# filter out completed participants
df_end_all = df_end_all[df_end_all['ParticipantIdentifier'].isin(complete_participant_ids)]

# drop duplicates(one row per participant per day)
df_end_all = df_end_all.groupby(['ParticipantIdentifier', 'date']).first().reset_index()


# fill missing (participant, date) in [date_min, date_max] with planning_prompt=0
# (must check per participant — date alone is wrong: another participant may already have that day)
fills = []
for participant_id in complete_participant_ids:
    min_date = summary_surveytask.loc[summary_surveytask['ParticipantIdentifier'] == participant_id, 'date_min'].iloc[0]
    max_date = summary_surveytask.loc[summary_surveytask['ParticipantIdentifier'] == participant_id, 'date_max'].iloc[0]
    existing = set(
        pd.to_datetime(
            df_end_all.loc[df_end_all['ParticipantIdentifier'] == participant_id, 'date'],
            errors='coerce',
        ).dt.date.dropna()
    )
    for ts in pd.date_range(pd.to_datetime(min_date), pd.to_datetime(max_date), freq='D'):
        d = ts.date()
        if d not in existing:
            fills.append(
                {'ParticipantIdentifier': participant_id, 'date': d, 'planning_prompt': 0}
            )
if fills:
    df_end_all = pd.concat([df_end_all, pd.DataFrame(fills)], ignore_index=True)
    df_end_all = df_end_all.sort_values(['ParticipantIdentifier', 'date']).reset_index(drop=True)

# print(df_end_all[df_end_all['participantidentifier'] == 31])
print(df_end_all)
# save the dataframe
df_end_all.to_csv(os.path.join(folder, 'df_end_all.csv'), index=False)



    ParticipantIdentifier        date      time  planning_prompt
0                     118  2025-09-14  19:00:11                1
1                     118  2025-09-15  19:00:33                1
2                     118  2025-09-16  19:00:12                0
3                     118  2025-09-17  19:00:13                0
4                     118  2025-09-18  19:00:12                0
..                    ...         ...       ...              ...
925                   225  2026-03-04  23:00:36                1
926                   225  2026-03-05  23:01:19                1
927                   225  2026-03-06  23:01:05                1
928                   225  2026-03-07  23:01:11                0
929                   225  2026-03-08  22:01:03                0

[930 rows x 4 columns]


In [242]:
# the above rows do not contain no delivery data, so we need to complete the data
# first, complete walking suggestions

questions = ["MessageDisplay With Template"]

tmp = (
    survey_results_flat
    .loc[survey_results_flat["ResultIdentifier"].isin(questions)]
    .copy()
)


tmp["datetime"] = pd.to_datetime(tmp["QuestionEndDate"], errors="coerce", utc=True)
s = tmp["QuestionEndDate"].astype(str).str.replace(
    r"([+-]\d{2}:\d{2}|Z)$", "", regex=True
)
tmp["datetime_local"] = pd.to_datetime(s, errors="coerce")


# Local date from original offset timestamp string (YYYY-MM-DD part)
tmp["date"] = tmp["datetime_local"].dt.date

tmp['time'] = tmp["datetime_local"].dt.time
print(len(tmp))


# check the date range of the walking suggestions for each participant
df_gif_all = []  # Use list for efficient appending

for n in range(len(complete_participant_ids)):
    participant_id = complete_participant_ids[n]
    gif_rows_participant = gif_rows.loc[gif_rows['ParticipantIdentifier'] == participant_id].copy()
    print(len(gif_rows_participant))
    gif_rows_participant['Timestamp'] = pd.to_datetime(gif_rows_participant['Timestamp'])    
    # gif_rows_open_participant = gif_rows_open.loc[gif_rows_open['ParticipantIdentifier'] == participant_id].copy()
    # gif_rows_open_participant['Timestamp'] = pd.to_datetime(gif_rows_open_participant['Timestamp'])
    tmp_participant = tmp.loc[tmp['ParticipantIdentifier'] == participant_id].copy()

    # extract the date and time from the timestamp
    gif_rows_participant['date'] = gif_rows_participant['Timestamp'].dt.date
    gif_rows_participant['time'] = gif_rows_participant['Timestamp'].dt.time
    # gif_rows_open_participant['date'] = gif_rows_open_participant['Timestamp'].dt.date
    # gif_rows_open_participant['time'] = gif_rows_open_participant['Timestamp'].dt.time

    min_date = summary_surveytask.loc[summary_surveytask['ParticipantIdentifier'] == participant_id, 'date_min'].iloc[0]
    min_date = pd.to_datetime(min_date).date()
    
    # max_date = max_date_list[n].date()
    date_range_length = 85
    # date_range_length_list[n]

    # read wakeup and bedtime data for the participant
    df_wakeup_bedtime_participant = df_wakeup_bedtime.loc[df_wakeup_bedtime['ParticipantIdentifier'] == participant_id].copy()


    for i in range(date_range_length):
        date = min_date + pd.Timedelta(days=i)
        # print(date)
        if df_wakeup_bedtime_participant.shape[0] == 1:
            weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[0]
            weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[0]
        else:
            change_date = df_wakeup_bedtime_participant['QuestionEndDate']
            weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[0]
            weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[0]
            
            for j in range(len(df_wakeup_bedtime_participant)):
                change_date = df_wakeup_bedtime_participant['QuestionEndDate'].iloc[j]
                
                # Check if this is the applicable period
                if j < len(df_wakeup_bedtime_participant) - 1:
                    next_change_date = df_wakeup_bedtime_participant['QuestionEndDate'].iloc[j + 1]
                    if change_date <= date < next_change_date:
                        weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[j]
                        weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[j]
                        break
                else:
                    # Last entry - applies from change_date onwards
                    if change_date <= date:
                        weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[j]
                        weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[j]
                        break
        
        # Determine wakeup time based on weekday/weekend
        is_weekday = date.weekday() < 5
        wakeup_time = weekday_wakeup if is_weekday else weekend_wakeup

        wake_td = pd.Timedelta(hours=wakeup_time.hour, minutes=wakeup_time.minute, seconds=wakeup_time.second)
        m_time = wake_td + pd.Timedelta(minutes=60)   # Morning: 1 hour after wake
        a_time = wake_td + pd.Timedelta(minutes=360)  # Afternoon: 6 hours after wake

        # search for date and time in gif_rows_participant
        gif_row = gif_rows_participant.loc[(gif_rows_participant['date'] == date)].copy()
        # gif_row_open = gif_rows_open_participant.loc[(gif_rows_open_participant['date'] == date)]
        tmp_par_date = tmp_participant.loc[(tmp_participant['date'] == date)].copy()
        # print(tmp_par_date.head())

        
        # valid_count = [
        #     df_hourly_pageview.loc[(df_hourly_pageview['participantidentifier'] == participant_id) &
        #                         (df_hourly_pageview['date'] == date) &
        #                         (df_hourly_pageview['decision_time'] == dt),
        #                         'valid_count'].iloc[0]
        #     for dt in (0, 1)
        # ]

        if gif_row.shape[0] == 2:
            decision_time = 0
            for time_val, timestamp_val in zip(gif_row['time'].values, gif_row['Timestamp'].values):
                # Check if opened within 1 hour
                # open_status = 0
                # for open_timestamp in gif_row_open['timestamp'].values:
                #     time_diff = abs((pd.to_datetime(open_timestamp) - pd.to_datetime(timestamp_val)).total_seconds() / 60)
                #     if time_diff <= 60:  # Within 60 minutes
                #         open_status = 1
                #         break

                # check if the user interacted with the notification
                
                if len(tmp_par_date['datetime_local'].values) >= 2:
                    interacted = 1
                else:
                    interacted = 0
                    for interaction in tmp_par_date['datetime_local'].values:
                        if interaction < timestamp_val:
                            continue
                        time_diff = pd.Timedelta(interaction - timestamp_val).total_seconds() / 60
                        # print(time_diff)
                        if decision_time == 0 and 0 <= time_diff <= 300:
                            interacted = 1
                            break
                        elif decision_time == 1 and 0 <= time_diff <= 600:
                            interacted = 1
                            break

                df_gif_all.append({
                    'ParticipantIdentifier': participant_id,
                    'Date': date,
                    'Time': time_val,
                    'DecisionTime': decision_time,
                    'WalkingSuggestion': 1,
                    'Interacted': interacted
                    # 'ViewStatus': min(open_status + int(valid_count[decision_time] > 0), 1)
                })
                decision_time += 1
        elif gif_row.shape[0] == 1:
            # One delivery - determine if it's morning or afternoon
            delivered_time = gif_row['time'].iloc[0]
            delivered_timestamp = gif_row['Timestamp'].iloc[0]

            # Check if opened within 1 hour
            # TODO: check if this is correct
            # open_status = 0
            # for open_timestamp in gif_row_open['timestamp'].values:
            #     time_diff = abs((pd.to_datetime(open_timestamp) - pd.to_datetime(delivered_timestamp)).total_seconds() / 60)
            #     if time_diff <= 60:  # Within 60 minutes
            #         open_status = 1
            #         break

            # check if the user interacted with the notification
            
            if len(tmp_par_date['datetime_local'].values) >= 1:
                interacted = 1
            else:
                interacted = 0
                for interaction in tmp_par_date['datetime_local'].values:
                    if interaction < timestamp_val:
                        continue
                    time_diff = pd.Timedelta(interaction - timestamp_val).total_seconds() / 60
                    if decision_time == 0 and 0 <= time_diff <= 300:
                        interacted = 1
                        break
                    elif decision_time == 1 and 0 <= time_diff <= 600:
                        interacted = 1
                        break

        
            # Convert time to timedelta for comparison
            delivered_td = pd.Timedelta(
                hours=delivered_time.hour, 
                minutes=delivered_time.minute, 
                seconds=delivered_time.second
            )
            
            # Check which slot it's closer to
            is_morning = abs((delivered_td - m_time).total_seconds()) < abs((delivered_td - a_time).total_seconds())
            
            if is_morning:
                # Delivered in morning, not in afternoon
                df_gif_all.append({
                    'ParticipantIdentifier': participant_id,
                    'Date': date,
                    'Time': delivered_time,
                    'DecisionTime': 0,
                    'WalkingSuggestion': 1,
                    'Interacted': interacted
                    # 'ViewStatus': min(open_status + int(valid_count[0] > 0), 1)
                })
                df_gif_all.append({
                    'ParticipantIdentifier': participant_id,
                    'Date': date,
                    'Time': (pd.Timestamp('2000-01-01') + a_time).time(),
                    'DecisionTime': 1,
                    'WalkingSuggestion': 0,
                    'Interacted': 0
                    # 'ViewStatus': int(valid_count[1] > 0)
                })
            else:
                # Delivered in afternoon, not in morning
                df_gif_all.append({
                    'ParticipantIdentifier': participant_id,
                    'Date': date,
                    'Time': (pd.Timestamp('2000-01-01') + m_time).time(),
                    'DecisionTime': 0,
                    'WalkingSuggestion': 0,
                    'Interacted': 0
                    # 'ViewStatus': int(valid_count[0] > 0)
                })
                df_gif_all.append({
                    'ParticipantIdentifier': participant_id,
                    'Date': date,
                    'Time': delivered_time,
                    'DecisionTime': 1,
                    'WalkingSuggestion': 1,
                    'Interacted': interacted
                    # 'ViewStatus': min(open_status + int(valid_count[1] > 0), 1)
                })
        else:
            # No deliveries - both slots empty
            df_gif_all.append({
                'ParticipantIdentifier': participant_id,
                'Date': date,
                'Time': (pd.Timestamp('2000-01-01') + m_time).time(),
                'DecisionTime': 0,
                'WalkingSuggestion': 0,
                'Interacted': 0
                # 'ViewStatus': int(valid_count[0] > 0)
            })
            df_gif_all.append({
                'ParticipantIdentifier': participant_id,
                'Date': date,
                'Time': (pd.Timestamp('2000-01-01') + a_time).time(),
                'DecisionTime': 1,
                'WalkingSuggestion': 0,
                'Interacted': 0
                # 'ViewStatus': int(valid_count[1] > 0)
            })

df_gif_all = pd.DataFrame(df_gif_all)

# add a column for the fraction of interacted=1 for the past 7 days
df_gif_all['Interacted_7d'] = df_gif_all.groupby('ParticipantIdentifier')['Interacted'].transform(lambda x: x.rolling(window=14, min_periods=14).mean())

print(df_gif_all)
print(df_gif_all.Interacted.value_counts())
# print(df_gif_all[df_gif_all['participantidentifier'] == 75].head(50))
# save the dataframe
df_gif_all.to_csv(os.path.join(folder, 'df_gif_all.csv'), index=False)


660
101
81
80
82
81
71
76
86
86
87
178
     ParticipantIdentifier        Date      Time  DecisionTime  \
0                      118  2025-09-14  08:30:00             0   
1                      118  2025-09-14  13:30:00             1   
2                      118  2025-09-15  08:30:10             0   
3                      118  2025-09-15  13:30:00             1   
4                      118  2025-09-16  08:30:13             0   
...                    ...         ...       ...           ...   
1865                   225  2026-03-07  12:30:00             1   
1866                   225  2026-03-08  07:30:00             0   
1867                   225  2026-03-08  12:30:00             1   
1868                   225  2026-03-09  07:30:00             0   
1869                   225  2026-03-09  12:30:00             1   

      WalkingSuggestion  Interacted  Interacted_7d  
0                     0           0            NaN  
1                     0           0            NaN  
2        

In [243]:
# second, complete salience messages
questions = ["Salience Message Display"]

tmp = (
    survey_results_flat
    .loc[survey_results_flat["ResultIdentifier"].isin(questions)]
    .copy()
)


tmp["datetime"] = pd.to_datetime(tmp["QuestionEndDate"], errors="coerce", utc=True)
s = tmp["QuestionEndDate"].astype(str).str.replace(
    r"([+-]\d{2}:\d{2}|Z)$", "", regex=True
)
tmp["datetime_local"] = pd.to_datetime(s, errors="coerce")


# Local date from original offset timestamp string (YYYY-MM-DD part)
tmp["date"] = tmp["datetime_local"].dt.date

tmp['time'] = tmp["datetime_local"].dt.time
print(len(tmp))
df_salience_all = []
# check the date range of the salience messages
for n in range(len(complete_participant_ids)):
    participant_id = complete_participant_ids[n]
    salience_rows_participant = salience_rows.loc[salience_rows['ParticipantIdentifier'] == participant_id].copy()
    salience_rows_participant['Timestamp'] = pd.to_datetime(salience_rows_participant['Timestamp'])
    # salience_rows_open_participant = salience_rows_open.loc[salience_rows_open['ParticipantIdentifier'] == participant_id].copy()
    # salience_rows_open_participant['Timestamp'] = pd.to_datetime(salience_rows_open_participant['Timestamp'])
    # print(f"Participant {participant_id} salience messages date range: {salience_rows_participant['timestamp'].min()} to {salience_rows_participant['timestamp'].max()}")
    tmp_participant = tmp.loc[tmp['ParticipantIdentifier'] == participant_id].copy()
    # extract the date and time from the timestamp
    salience_rows_participant['date'] = salience_rows_participant['Timestamp'].dt.date
    salience_rows_participant['time'] = salience_rows_participant['Timestamp'].dt.time
    # salience_rows_open_participant['date'] = salience_rows_open_participant['timestamp'].dt.date
    # salience_rows_open_participant['time'] = salience_rows_open_participant['timestamp'].dt.time

    min_date = summary_surveytask.loc[summary_surveytask['ParticipantIdentifier'] == participant_id, 'date_min'].iloc[0]
    min_date = pd.to_datetime(min_date).date()

    date_range_length = 85
    # date_range_length_list[n]

    for i in range(date_range_length):
        date = min_date + pd.Timedelta(days=i)
        salience_row = salience_rows_participant.loc[(salience_rows_participant['date'] == date)]
        tmp_par_date = tmp_participant.loc[(tmp_participant['date'] == date)].copy()
        # salience_row_open = salience_rows_open_participant.loc[(salience_rows_open_participant['date'] == date)]

        # open_status = 0
        # for open_timestamp in salience_row_open['timestamp'].values:
        #     time_diff = abs((pd.to_datetime(open_timestamp) - pd.to_datetime(salience_row['timestamp'].values[0])).total_seconds() / 60)
        #     if time_diff <= 60:  # Within 60 minutes
        #         open_status = 1
        #         break

        if len(tmp_par_date['datetime_local'].values) >= 1:
            interacted = 1
        else:
            interacted = 0

        if salience_row.shape[0] == 1:
            df_salience_all.append({
                'ParticipantIdentifier': participant_id,
                'Date': date,
                'Time': salience_row['time'].iloc[0],
                'SalienceMessage': 1,
                'Interacted': interacted
                # 'open': open_status
            })
        else:
            df_salience_all.append({
                'ParticipantIdentifier': participant_id,
                'Date': date,
                'Time': (pd.Timestamp('2000-01-01') + pd.Timedelta(hours=11, minutes=45)).time(),
                'SalienceMessage': 0,
                'Interacted': interacted
                # 'open': 0
            })

df_salience_all = pd.DataFrame(df_salience_all)
print(df_salience_all)
print(df_salience_all.Interacted.value_counts())

# add a column for the fraction of interacted=1 for the past 7 days
df_salience_all['Interacted_7d'] = df_salience_all.groupby('ParticipantIdentifier')['Interacted'].transform(lambda x: x.rolling(window=7, min_periods=7).mean())

# print(df_salience_all.head(85))
# print(df_salience_all.open.value_counts())

# save the dataframe
df_salience_all.to_csv(os.path.join(folder, 'df_salience_all.csv'), index=False)


420
    ParticipantIdentifier        Date      Time  SalienceMessage  Interacted
0                     118  2025-09-14  11:45:00                0           0
1                     118  2025-09-15  11:45:11                1           1
2                     118  2025-09-16  11:45:00                0           0
3                     118  2025-09-17  11:45:11                1           1
4                     118  2025-09-18  11:45:07                1           1
..                    ...         ...       ...              ...         ...
930                   225  2026-03-05  11:45:00                0           1
931                   225  2026-03-06  11:45:00                0           1
932                   225  2026-03-07  11:45:00                0           1
933                   225  2026-03-08  11:45:00                0           1
934                   225  2026-03-09  11:45:00                0           0

[935 rows x 5 columns]
Interacted
0    532
1    403
Name: count, dtype:

## Check missing data in step counts (not wearing fitbit for greater than 8 hours)
- case 1- missing hours: user receive walking suggestions at 9am, but only started wearing fitbit until 10 am 
  treatment: 
- case 2- missing days: user didn't wear fitbit for most hours between the wakeup and bedtime

In [244]:
# using heartrate to define drop out...etc
heartratebymin = pd.read_csv(folder/ 'filtered_activities-heart.csv')
print(heartratebymin.head())

#change to local time
heartratebymin = convert_utc_columns_to_user_local(
    heartratebymin,
    datetime_cols=["DateTime"],
    participant_col="ParticipantIdentifier",
    join_date_col="DateTime",
)
#filter out completed users

                    DateTime               InsertedDate ParticipantIdentifier  \
0  2025-09-11T00:00:00+00:00  2025-09-12T19:28:00+00:00                   106   
1  2025-09-11T00:01:00+00:00  2025-09-12T19:28:00+00:00                   106   
2  2025-09-11T00:02:00+00:00  2025-09-12T19:28:00+00:00                   106   
3  2025-09-11T00:03:00+00:00  2025-09-12T19:28:00+00:00                   106   
4  2025-09-11T00:04:00+00:00  2025-09-12T19:28:00+00:00                   106   

   Value                    _source_file _source_folder  
0  65.48  filtered_activities-heart.json     2025-09-13  
1  63.77  filtered_activities-heart.json     2025-09-13  
2  61.64  filtered_activities-heart.json     2025-09-13  
3  62.19  filtered_activities-heart.json     2025-09-13  
4  61.25  filtered_activities-heart.json     2025-09-13  


In [245]:
# filter out completed users
heartratebymin = heartratebymin[heartratebymin['ParticipantIdentifier'].isin(complete_participant_ids)].copy()
print(heartratebymin.head())

               DateTime               InsertedDate ParticipantIdentifier  \
951 2025-09-11 09:05:00  2025-09-12T16:08:00+00:00                   118   
952 2025-09-11 09:06:00  2025-09-12T16:08:00+00:00                   118   
953 2025-09-11 09:07:00  2025-09-12T16:08:00+00:00                   118   
954 2025-09-11 09:08:00  2025-09-12T16:08:00+00:00                   118   
955 2025-09-11 09:09:00  2025-09-12T16:08:00+00:00                   118   

     Value                    _source_file _source_folder  
951  87.38  filtered_activities-heart.json     2025-09-13  
952  85.96  filtered_activities-heart.json     2025-09-13  
953  77.88  filtered_activities-heart.json     2025-09-13  
954  78.58  filtered_activities-heart.json     2025-09-13  
955  81.33  filtered_activities-heart.json     2025-09-13  


In [246]:
# for the active phase, we require 12*7 = 84 days of step count data
heartratebymin['DateTime'] = pd.to_datetime(heartratebymin['DateTime'])
heartratebymin['Date'] = heartratebymin['DateTime'].dt.date

# sort by participantidentifier and date
heartratebymin = heartratebymin.sort_values(by=['ParticipantIdentifier', 'DateTime'])

summary = (
    heartratebymin
    .groupby('ParticipantIdentifier')['Date']
    .agg(date_min='min', date_max='max', n_days='nunique')
    .assign(span_days=lambda df: (pd.to_datetime(df.date_max) -
                                  pd.to_datetime(df.date_min)).dt.days + 1)
)



# make participantidentifier a column name
summary.reset_index(inplace=True)
summary.rename(columns={'ParticipantIdentifier': 'ParticipantIdentifier'}, inplace=True)

print(summary)

   ParticipantIdentifier    date_min    date_max  n_days  span_days
0                    118  2025-09-08  2025-12-08      92         92
1                    141  2025-10-13  2026-01-14      94         94
2                    143  2025-08-31  2025-12-13      94        105
3                    151  2025-09-19  2026-01-14     112        118
4                    160  2025-11-23  2026-02-28      98         98
5                    170  2025-09-26  2026-01-02      89         99
6                    184  2025-10-17  2026-01-31     103        107
7                    188  2025-10-21  2026-03-12     138        143
8                    195  2025-11-26  2026-03-02      90         97
9                    204  2025-11-07  2026-02-07      93         93
10                   225  2025-11-26  2026-03-07     102        102


In [247]:
# Select date range based on daily survey start and end date
heartratebymin_selected = []
for participant_id in complete_participant_ids:
    heartratebymin_participant = heartratebymin[heartratebymin['ParticipantIdentifier'] == participant_id]
    summary_row = summary_surveytask.loc[
        summary_surveytask['ParticipantIdentifier'] == participant_id
    ].iloc[0]  # assumes one row per participant

   
    start_date = pd.to_datetime(summary_row['date_min']).date() - pd.Timedelta(days=7)
    end_date = start_date + pd.Timedelta(days=85)
    heartratebymin_participant = heartratebymin_participant[heartratebymin_participant.Date >= start_date]
    heartratebymin_participant = heartratebymin_participant[heartratebymin_participant.Date <= end_date]
    heartratebymin_selected.append(heartratebymin_participant)

heartratebymin_selected = pd.concat(heartratebymin_selected)
# print(heartratebymin_selected.head())

heartratebymin_selected = heartratebymin_selected[["ParticipantIdentifier", "DateTime", "Value", "Date"]]
print(heartratebymin_selected[heartratebymin_selected['ParticipantIdentifier'] == "118"])

        ParticipantIdentifier            DateTime   Value        Date
91280                     118 2025-09-08 09:10:00   95.89  2025-09-08
91281                     118 2025-09-08 09:11:00   86.73  2025-09-08
91282                     118 2025-09-08 09:12:00   76.26  2025-09-08
91283                     118 2025-09-08 09:13:00   73.54  2025-09-08
91284                     118 2025-09-08 09:14:00   72.39  2025-09-08
...                       ...                 ...     ...         ...
1245892                   118 2025-12-01 18:50:00  103.85  2025-12-01
1245893                   118 2025-12-01 18:51:00  107.00  2025-12-01
1245894                   118 2025-12-01 18:52:00   97.81  2025-12-01
1245895                   118 2025-12-01 18:53:00   96.24  2025-12-01
1245896                   118 2025-12-01 18:54:00   92.50  2025-12-01

[78959 rows x 4 columns]


In [248]:
# read step count data by minute
stepcountbymin = pd.read_csv(folder /'filtered_activities-steps.csv')
stepcountbymin.head()

# change to local time
stepcountbymin = convert_utc_columns_to_user_local(
    stepcountbymin,
    datetime_cols=["DateTime"],
    participant_col="ParticipantIdentifier",
    join_date_col="DateTime",
)

/var/folders/hz/q4hdsnpj1h50x_jpx8v5mrvc0000gn/T/ipykernel_83343/1237549820.py:2: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  stepcountbymin = pd.read_csv(folder /'filtered_activities-steps.csv')


In [249]:
# for the active phase, we require 12*7 = 84 days of step count data
stepcountbymin['DateTime'] = pd.to_datetime(stepcountbymin['DateTime'])
stepcountbymin['Date'] = stepcountbymin['DateTime'].dt.date

# filter out the data for complete participants
stepcountbymin = stepcountbymin[stepcountbymin['ParticipantIdentifier'].isin(complete_participant_ids)]

summary = (
    stepcountbymin
    .groupby('ParticipantIdentifier')['Date']
    .agg(date_min='min', date_max='max', n_days='nunique')
    .assign(span_days=lambda df: (pd.to_datetime(df.date_max) -
                                  pd.to_datetime(df.date_min)).dt.days + 1)
)



# make participantidentifier a column name
summary.reset_index(inplace=True)
summary.rename(columns={'ParticipantIdentifier': 'ParticipantIdentifier'}, inplace=True)

print(summary)

   ParticipantIdentifier    date_min    date_max  n_days  span_days
0                    118  2025-09-11  2025-12-09      90         90
1                    141  2025-09-09  2026-01-14     128        128
2                    143  2025-08-11  2025-12-14     126        126
3                    151  2025-09-14  2026-01-14     123        123
4                    160  2025-10-20  2026-03-04     136        136
5                    170  2025-09-25  2026-01-09     107        107
6                    184  2025-09-18  2026-01-31     136        136
7                    188  2025-10-21  2026-03-12     143        143
8                    195  2025-10-22  2026-03-02     132        132
9                    204  2025-11-05  2026-02-07      95         95
10                   225  2025-10-27  2026-03-07     132        132


In [250]:
# Select date range based on daily survey start and end date
stepcountbymin_selected = []
for participant_id in complete_participant_ids:
    print(participant_id)
    stepcountbymin_participant = stepcountbymin[stepcountbymin['ParticipantIdentifier'] == participant_id]
    summary_row = summary_surveytask.loc[
        summary_surveytask['ParticipantIdentifier'] == participant_id
    ].iloc[0]  # assumes one row per participant

    start_date = pd.to_datetime(summary_row['date_min']).date() - pd.Timedelta(days=7)
    end_date = start_date + pd.Timedelta(days=85)
    print(start_date, end_date)
    stepcountbymin_participant = stepcountbymin_participant[stepcountbymin_participant.Date >= start_date]
    stepcountbymin_participant = stepcountbymin_participant[stepcountbymin_participant.Date <= end_date]
    stepcountbymin_selected.append(stepcountbymin_participant)

stepcountbymin_selected = pd.concat(stepcountbymin_selected)
stepcountbymin_selected = stepcountbymin_selected[["ParticipantIdentifier", "DateTime", "Value", "Date"]]
print(stepcountbymin_selected[stepcountbymin_selected['ParticipantIdentifier'] == "118"])

118
2025-09-07 2025-12-01
141
2025-10-13 2026-01-06
143
2025-09-15 2025-12-09
151
2025-09-22 2025-12-16
160
2025-11-24 2026-02-17
170
2025-09-29 2025-12-23
184
2025-10-26 2026-01-19
188
2025-11-24 2026-02-17
195
2025-12-02 2026-02-25
204
2025-11-10 2026-02-03
225
2025-12-08 2026-03-03
        ParticipantIdentifier            DateTime  Value        Date
7462                      118 2025-09-11 00:00:00      0  2025-09-11
7463                      118 2025-09-11 00:01:00      0  2025-09-11
7464                      118 2025-09-11 00:02:00      0  2025-09-11
7465                      118 2025-09-11 00:03:00      0  2025-09-11
7466                      118 2025-09-11 00:04:00      0  2025-09-11
...                       ...                 ...    ...         ...
4512092                   118 2025-12-01 23:55:00      0  2025-12-01
4512093                   118 2025-12-01 23:56:00      0  2025-12-01
4512094                   118 2025-12-01 23:57:00      0  2025-12-01
4512095                 

In [251]:
# checck missing days by filtering out the days with less than 8 hours of wearing fitbit (more than 8 hours of heart rate =0 or nan
# within the wakeup and bedtime)

missing_days_list = []
for participant_id in complete_participant_ids:
    heartratebymin_participant = heartratebymin_selected.loc[heartratebymin_selected['ParticipantIdentifier'] == participant_id].copy()
    stepcountbymin_participant = stepcountbymin_selected.loc[stepcountbymin_selected['ParticipantIdentifier'] == participant_id].copy()

    df_wakeup_bedtime_participant = df_wakeup_bedtime.loc[df_wakeup_bedtime['ParticipantIdentifier'] == participant_id].copy()

    # we minus 7 day becasue we want to include past 7 days data of day 1
    min_date = summary_surveytask.loc[
        summary_surveytask['ParticipantIdentifier'] == participant_id
    ].iloc[0].date_min - pd.Timedelta(days=7)
    date_range_length = 92

    for i in range(date_range_length):
        date = min_date + pd.Timedelta(days=i)
        # print(date)
        if df_wakeup_bedtime_participant.shape[0] == 1:
            weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[0]
            weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[0]
            weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[0]
            weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[0]
        else:
            change_date = df_wakeup_bedtime_participant['QuestionEndDate']
            weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[0]
            weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[0]
            weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[0]
            weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[0]
            
            for j in range(len(df_wakeup_bedtime_participant)):
                change_date = df_wakeup_bedtime_participant['QuestionEndDate'].iloc[j]
                
                # Check if this is the applicable period
                if j < len(df_wakeup_bedtime_participant) - 1:
                    next_change_date = df_wakeup_bedtime_participant['QuestionEndDate'].iloc[j + 1]
                    if change_date <= date < next_change_date:
                        weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[j]
                        weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[j]
                        weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[j]
                        weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[j]
                        break
                else:
                    # Last entry - applies from change_date onwards
                    if change_date <= date:
                        weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[j]
                        weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[j]
                        weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[j]
                        weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[j]
                        break
        
        # Determine wakeup time based on weekday/weekend
        is_weekday = date.weekday() < 5
        wakeup_time = weekday_wakeup if is_weekday else weekend_wakeup
        bedtime_time = weekday_bedtime if is_weekday else weekend_bedtime


        # filter out the days with less than 8 hours of wearing fitbit (more than 8 hours of heart rate =0 or nan
        # within the wakeup and bedtime)
        heartratebymin_participant_date = heartratebymin_participant[heartratebymin_participant.Date == date]
        heartratebymin_participant_date = heartratebymin_participant_date[heartratebymin_participant_date.DateTime.dt.time >= wakeup_time]
        heartratebymin_participant_date = heartratebymin_participant_date[heartratebymin_participant_date.DateTime.dt.time <= bedtime_time]
        
        stepcountbymin_participant_date = stepcountbymin_participant[stepcountbymin_participant.Date == date]
        stepcountbymin_participant_date = stepcountbymin_participant_date[stepcountbymin_participant_date.DateTime.dt.time >= wakeup_time]
        stepcountbymin_participant_date = stepcountbymin_participant_date[stepcountbymin_participant_date.DateTime.dt.time <= bedtime_time]

        # Filter valid heart rate readings (not 0 and not NaN)
        # Build the mask on heartrate rows only to avoid index misalignment warnings.
        step_active_times = stepcountbymin_participant_date.loc[
            (stepcountbymin_participant_date['Value'] > 0) &
            (stepcountbymin_participant_date['Value'].notna()),
            'DateTime'
        ]
        valid_hr = heartratebymin_participant_date[
            ((heartratebymin_participant_date['Value'] != 0) &
             (heartratebymin_participant_date['Value'].notna())) |
            (heartratebymin_participant_date['DateTime'].isin(step_active_times))
        ]
        
        # Calculate time span from first to last valid reading
        if len(valid_hr) > 0:
            first_timestamp = valid_hr['DateTime'].min()
            last_timestamp = valid_hr['DateTime'].max()
            time_span = last_timestamp - first_timestamp
            valid_hours = time_span.total_seconds() / 3600
        else:
            valid_hours = 0

        wearing = 0 if valid_hours < 8 else 1


        # Mark as missing if less than 8 hours of valid data

        missing_days_list.append({
            'ParticipantIdentifier': participant_id,
            'Date': date,
            'DayWearing': wearing,
            'ValidHours': valid_hours
        })

df_missing_days = pd.DataFrame(missing_days_list)

# add a column of past 7 days daywearing
df_missing_days['past7days_daywearing'] = df_missing_days['DayWearing'].ewm(span=7, min_periods=1).mean()

# add a column of next day wearing
df_missing_days['nextday_wearing'] = df_missing_days['DayWearing'].shift(-1)

df_missing_days.to_csv(os.path.join(folder, 'missing_days.csv'), index=False)


In [252]:
print(df_missing_days[df_missing_days['ParticipantIdentifier'] == "219"][51:100])

Empty DataFrame
Columns: [ParticipantIdentifier, Date, DayWearing, ValidHours, past7days_daywearing, nextday_wearing]
Index: []


In [253]:
## extract hourly level missingness

# check missing hours by filtering out hours with less than a certain threshold of valid heart rate data
# within the wakeup and bedtime)

missing_hours_list = []

for participant_id in complete_participant_ids:
    heartratebymin_participant = heartratebymin_selected[heartratebymin_selected['ParticipantIdentifier'] == participant_id].copy()
    stepcountbymin_participant = stepcountbymin_selected.loc[stepcountbymin_selected['ParticipantIdentifier'] == participant_id].copy()
    df_wakeup_bedtime_participant = df_wakeup_bedtime.loc[df_wakeup_bedtime['ParticipantIdentifier'] == participant_id].copy()

    # we minus 7 day because we want to include past 7 days data of day 1
    # TODO: figured out we may not need to minus 1 day because we start modeling step counts after the first day of end of day survey
    min_date = summary_surveytask.loc[
        summary_surveytask['ParticipantIdentifier'] == participant_id
    ].iloc[0].date_min - pd.Timedelta(days=7)
    date_range_length = 92

    for i in range(date_range_length):
        date = min_date + pd.Timedelta(days=i)
        
        if df_wakeup_bedtime_participant.shape[0] == 1:
            weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[0]
            weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[0]
            weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[0]
            weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[0]
        else:
            change_date = df_wakeup_bedtime_participant['QuestionEndDate']
            weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[0]
            weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[0]
            weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[0]
            weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[0]
            
            for j in range(len(df_wakeup_bedtime_participant)):
                change_date = df_wakeup_bedtime_participant['QuestionEndDate'].iloc[j]
                
                # Check if this is the applicable period
                if j < len(df_wakeup_bedtime_participant) - 1:
                    next_change_date = df_wakeup_bedtime_participant['QuestionEndDate'].iloc[j + 1]
                    if change_date <= date < next_change_date:
                        weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[j]
                        weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[j]
                        weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[j]
                        weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[j]
                        break
                else:
                    # Last entry - applies from change_date onwards
                    if change_date <= date:
                        weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[j]
                        weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[j]
                        weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[j]
                        weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[j]
                        break
        
        # Determine wakeup time based on weekday/weekend
        is_weekday = date.weekday() < 5
        wakeup_time = weekday_wakeup if is_weekday else weekend_wakeup
        bedtime_time = weekday_bedtime if is_weekday else weekend_bedtime

        # Get all heart rate data for this date
        heartratebymin_participant_date = heartratebymin_participant[heartratebymin_participant.Date == date]
        heartratebymin_participant_date = heartratebymin_participant_date[heartratebymin_participant_date.DateTime.dt.time >= wakeup_time]
        heartratebymin_participant_date = heartratebymin_participant_date[heartratebymin_participant_date.DateTime.dt.time <= bedtime_time]

        stepcountbymin_participant_date = stepcountbymin_participant[stepcountbymin_participant.Date == date]
        stepcountbymin_participant_date = stepcountbymin_participant_date[stepcountbymin_participant_date.DateTime.dt.time >= wakeup_time]
        stepcountbymin_participant_date = stepcountbymin_participant_date[stepcountbymin_participant_date.DateTime.dt.time <= bedtime_time]
        
        # Create hourly bins from wakeup to bedtime
        wakeup_datetime = pd.Timestamp.combine(date, wakeup_time)
        bedtime_datetime = pd.Timestamp.combine(date, bedtime_time)
        
        # wake up set to floor
        # anchor_date = pd.Timestamp('2000-01-01')  
        # wakeup_floor = (
        # pd.Timestamp.combine(anchor_date, wakeup_time)
        # .floor('H')                       
        # .time()
        # )

        # bedtime set to ceil
        # bedtime_ceil = (
        # pd.Timestamp.combine(anchor_date, bedtime_time)
        # .ceil('H')                       
        # .time()
        # )
        
        # Generate hourly time bins
        num_decisions = 2 # 2 decision points per day wakeup + 1, wakeup + 6
        current_decision = 0
        while current_decision < num_decisions:
            start_window = wakeup_datetime + pd.Timedelta(hours=1) if current_decision == 0 else wakeup_datetime + pd.Timedelta(hours=6)
            end_window = start_window + pd.Timedelta(hours=4)
            
            # Filter data for this hour
            hour_data = heartratebymin_participant_date[
                (heartratebymin_participant_date['DateTime'] >= start_window) &
                (heartratebymin_participant_date['DateTime'] < end_window)
            ]
            
            step_active_times = stepcountbymin_participant_date.loc[
                (stepcountbymin_participant_date['Value'] > 0) &
                (stepcountbymin_participant_date['Value'].notna()),
                'DateTime'
            ]

            # Filter valid heart rate readings (not 0 and not NaN)
            valid_hr = hour_data[
                (hour_data['Value'] != 0) & 
                (hour_data['Value'].notna()) | 
                (hour_data['DateTime'].isin(step_active_times))
            ]
            
            # Calculate time span of valid data in this hour
            if len(valid_hr) > 0:
                first_timestamp = valid_hr['DateTime'].min()
                last_timestamp = valid_hr['DateTime'].max()
                time_span = last_timestamp - first_timestamp
                valid_minutes = time_span.total_seconds() / 60
            else:
                valid_minutes = 0
            
            # Mark as missing if less than a threshold (e.g., 30 minutes of valid data in the hour)
            # You can adjust this threshold as needed
            wearing = 0 if valid_minutes < 200 else 1
            
            missing_hours_list.append({
                'ParticipantIdentifier': participant_id,
                'Date': date,
                'DecisionTime': current_decision,
                'DateTimeStart': start_window,
                'HourWearing': wearing
            })
            
            current_decision += 1

df_missing_hours = pd.DataFrame(missing_hours_list)
print(df_missing_hours[df_missing_hours['ParticipantIdentifier'] == "219"])

# save the dataframe
df_missing_hours.to_csv(os.path.join(folder, 'missing_hours.csv'), index=False)


Empty DataFrame
Columns: [ParticipantIdentifier, Date, DecisionTime, DateTimeStart, HourWearing]
Index: []


In [254]:
## extract 2 hours level missingness

# check missing hours by filtering out hours with less than a certain threshold of valid heart rate data
# within the wakeup and bedtime)

missing_2hours_list = []

for participant_id in complete_participant_ids:
    heartratebymin_participant = heartratebymin_selected[heartratebymin_selected['ParticipantIdentifier'] == participant_id].copy()
    stepcountbymin_participant = stepcountbymin_selected.loc[stepcountbymin_selected['ParticipantIdentifier'] == participant_id].copy()

    df_wakeup_bedtime_participant = df_wakeup_bedtime.loc[df_wakeup_bedtime['ParticipantIdentifier'] == participant_id].copy()

    # we minus 1 day because we want to include yesterday's data of day 1
    # TODO: figured out we may not need to minus 1 day because we start modeling step counts after the first day of end of day survey
    min_date = summary_surveytask.loc[
        summary_surveytask['ParticipantIdentifier'] == participant_id
    ].iloc[0].date_min 
    date_range_length = 85

    for i in range(date_range_length):
        date = min_date + pd.Timedelta(days=i)
        
        if df_wakeup_bedtime_participant.shape[0] == 1:
            weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[0]
            weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[0]
            weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[0]
            weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[0]
        else:
            change_date = df_wakeup_bedtime_participant['QuestionEndDate']
            weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[0]
            weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[0]
            weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[0]
            weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[0]
            
            for j in range(len(df_wakeup_bedtime_participant)):
                change_date = df_wakeup_bedtime_participant['QuestionEndDate'].iloc[j]
                
                # Check if this is the applicable period
                if j < len(df_wakeup_bedtime_participant) - 1:
                    next_change_date = df_wakeup_bedtime_participant['QuestionEndDate'].iloc[j + 1]
                    if change_date <= date < next_change_date:
                        weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[j]
                        weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[j]
                        weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[j]
                        weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[j]
                        break
                else:
                    # Last entry - applies from change_date onwards
                    if change_date <= date:
                        weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[j]
                        weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[j]
                        weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[j]
                        weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[j]
                        break
        
        # Determine wakeup time based on weekday/weekend
        is_weekday = date.weekday() < 5
        wakeup_time = weekday_wakeup if is_weekday else weekend_wakeup
        bedtime_time = weekday_bedtime if is_weekday else weekend_bedtime

        # Get all heart rate data for this date
        heartratebymin_participant_date = heartratebymin_participant[heartratebymin_participant.Date == date]
        # heartratebymin_participant_date = heartratebymin_participant_date[heartratebymin_participant_date.DateTime.dt.time >= wakeup_time]
        heartratebymin_participant_date = heartratebymin_participant_date[heartratebymin_participant_date.DateTime.dt.time <= bedtime_time]

        stepcountbymin_participant_date = stepcountbymin_participant[stepcountbymin_participant.Date == date]
        # stepcountbymin_participant_date = stepcountbymin_participant_date[stepcountbymin_participant_date.DateTime.dt.time >= wakeup_time]
        stepcountbymin_participant_date = stepcountbymin_participant_date[stepcountbymin_participant_date.DateTime.dt.time <= bedtime_time]
        
        # Create hourly bins from wakeup to bedtime
        wakeup_datetime = pd.Timestamp.combine(date, wakeup_time)
        bedtime_datetime = pd.Timestamp.combine(date, bedtime_time)
        
        
        # Generate hourly time bins
        num_decisions = 2 # 2 decision points per day wakeup + 1, wakeup + 6
        current_decision = 0
        while current_decision < num_decisions:
            end_window = wakeup_datetime + pd.Timedelta(hours=1) if current_decision == 0 else wakeup_datetime + pd.Timedelta(hours=6)
            start_window = end_window - pd.Timedelta(hours=2)

            # print(start_window, end_window)
            
            # Filter data for this hour
            hour_data = heartratebymin_participant_date[
                (heartratebymin_participant_date['DateTime'] >= start_window) &
                (heartratebymin_participant_date['DateTime'] < end_window)
            ]
            
            step_active_times = stepcountbymin_participant_date.loc[
                (stepcountbymin_participant_date['Value'] > 0) &
                (stepcountbymin_participant_date['Value'].notna()),
                'DateTime'
            ]

            # Filter valid heart rate readings (not 0 and not NaN)
            valid_hr = hour_data[
                (hour_data['Value'] != 0) & 
                (hour_data['Value'].notna()) |
                (hour_data['DateTime'].isin(step_active_times))
            ]
            # print(valid_hr)
            
            # Calculate time span of valid data in this hour
            if len(valid_hr) > 0:
                first_timestamp = valid_hr['DateTime'].min()
                last_timestamp = valid_hr['DateTime'].max()
                time_span = last_timestamp - first_timestamp
                valid_minutes = time_span.total_seconds() / 60
            else:
                valid_minutes = 0
            
            # Mark as missing if less than a threshold (e.g., 30 minutes of valid data in the hour)
            # You can adjust this threshold as needed
            wearing = 0 if valid_minutes <= 100 else 1
            
            missing_2hours_list.append({
                'ParticipantIdentifier': participant_id,
                'Date': date,
                'DecisionTime': current_decision,
                # 'DateTimeStart': start_window,
                'HourWearing': wearing
            })
            
            current_decision += 1

df_2hours = pd.DataFrame(missing_2hours_list)
# print(df_30_minutes[df_30_minutes['ParticipantIdentifier'] == 13].head())

print(df_2hours)

# print the proportion of Hourwearing =0
prop = df_2hours['HourWearing'].value_counts() / len(df_2hours)
print(prop)

# save the dataframe
df_2hours.to_csv(os.path.join(folder, 'missing_2hours.csv'), index=False)


     ParticipantIdentifier        Date  DecisionTime  HourWearing
0                      118  2025-09-14             0            1
1                      118  2025-09-14             1            1
2                      118  2025-09-15             0            1
3                      118  2025-09-15             1            1
4                      118  2025-09-16             0            1
...                    ...         ...           ...          ...
1865                   225  2026-03-07             1            0
1866                   225  2026-03-08             0            0
1867                   225  2026-03-08             1            0
1868                   225  2026-03-09             0            0
1869                   225  2026-03-09             1            0

[1870 rows x 4 columns]
HourWearing
1    0.840642
0    0.159358
Name: count, dtype: float64


In [255]:
# generate a 2x2 table of wearing in the morning (wakeup-1 hour to wakeup+1 hour)
# vs wearing in the rest of that day (after wakeup+1 hour to bedtime)

def _to_time(x):
    if pd.isna(x):
        return None
    if isinstance(x, pd.Timestamp):
        return x.time()
    if hasattr(x, 'hour') and hasattr(x, 'minute'):
        return x
    t = pd.to_datetime(str(x), errors='coerce')
    return None if pd.isna(t) else t.time()


def _get_wakeup_bedtime(df_wb_participant, date):
    if df_wb_participant.empty:
        return None, None

    wb = df_wb_participant.copy()
    wb['QuestionEndDate'] = pd.to_datetime(wb['QuestionEndDate'], errors='coerce').dt.normalize()

    for col in ['WeekdayWakeup', 'WeekendWakeup', 'WeekdayBedtime', 'WeekendBedtime']:
        wb[col] = wb[col].apply(_to_time)

    wb = wb.sort_values('QuestionEndDate', na_position='last').reset_index(drop=True)

    # Start from first available schedule values
    weekday_wakeup = wb['WeekdayWakeup'].dropna().iloc[0] if wb['WeekdayWakeup'].notna().any() else None
    weekend_wakeup = wb['WeekendWakeup'].dropna().iloc[0] if wb['WeekendWakeup'].notna().any() else None
    weekday_bedtime = wb['WeekdayBedtime'].dropna().iloc[0] if wb['WeekdayBedtime'].notna().any() else None
    weekend_bedtime = wb['WeekendBedtime'].dropna().iloc[0] if wb['WeekendBedtime'].notna().any() else None

    if any(t is None for t in [weekday_wakeup, weekend_wakeup, weekday_bedtime, weekend_bedtime]):
        return None, None

    wb_dated = wb.dropna(subset=['QuestionEndDate']).reset_index(drop=True)

    for j in range(len(wb_dated)):
        change_date = wb_dated['QuestionEndDate'].iloc[j]
        if j < len(wb_dated) - 1:
            next_change_date = wb_dated['QuestionEndDate'].iloc[j + 1]
            if change_date <= date < next_change_date:
                weekday_wakeup = wb_dated['WeekdayWakeup'].iloc[j] or weekday_wakeup
                weekend_wakeup = wb_dated['WeekendWakeup'].iloc[j] or weekend_wakeup
                weekday_bedtime = wb_dated['WeekdayBedtime'].iloc[j] or weekday_bedtime
                weekend_bedtime = wb_dated['WeekendBedtime'].iloc[j] or weekend_bedtime
                break
        else:
            if change_date <= date:
                weekday_wakeup = wb_dated['WeekdayWakeup'].iloc[j] or weekday_wakeup
                weekend_wakeup = wb_dated['WeekendWakeup'].iloc[j] or weekend_wakeup
                weekday_bedtime = wb_dated['WeekdayBedtime'].iloc[j] or weekday_bedtime
                weekend_bedtime = wb_dated['WeekendBedtime'].iloc[j] or weekend_bedtime
                break

    is_weekday = date.weekday() < 5
    wakeup_time = weekday_wakeup if is_weekday else weekend_wakeup
    bedtime_time = weekday_bedtime if is_weekday else weekend_bedtime
    return wakeup_time, bedtime_time


def _valid_minutes(hr_df, step_df):
    # A minute is treated as wearable if HR is valid, or step count is positive.
    step_active_times = step_df.loc[
        (step_df['Value'] > 0) & (step_df['Value'].notna()),
        'DateTime'
    ]
    valid_hr = hr_df.loc[
        ((hr_df['Value'] != 0) & (hr_df['Value'].notna())) |
        (hr_df['DateTime'].isin(step_active_times))
    ]

    if len(valid_hr) > 0:
        first_timestamp = valid_hr['DateTime'].min()
        last_timestamp = valid_hr['DateTime'].max()
        time_span = last_timestamp - first_timestamp
        return time_span.total_seconds() / 60
    return 0


# Tune this threshold if needed
RESTDAY_MIN_VALID_MINUTES = 200  # rest-of-day window

if 'df_2hours' not in globals():
    raise ValueError('Run the first block that creates df_2hours before this 2x2 table block.')

# Use morning wearing directly from block 1 (DecisionTime == 0)
morning_lookup = (
    df_2hours.loc[df_2hours['DecisionTime'] == 0, ['ParticipantIdentifier', 'Date', 'HourWearing']]
    .copy()
)
morning_lookup['Date'] = pd.to_datetime(morning_lookup['Date'], errors='coerce').dt.normalize()
morning_lookup = (
    morning_lookup.dropna(subset=['Date'])
    .groupby(['ParticipantIdentifier', 'Date'], as_index=False)['HourWearing']
    .max()
    .rename(columns={'HourWearing': 'morning_wearing'})
)
morning_lookup = morning_lookup.set_index(['ParticipantIdentifier', 'Date'])['morning_wearing']

wear_rows = []

for participant_id in complete_participant_ids:
    hr_p = heartratebymin_selected.loc[
        heartratebymin_selected['ParticipantIdentifier'] == participant_id
    ].copy()
    step_p = stepcountbymin_selected.loc[
        stepcountbymin_selected['ParticipantIdentifier'] == participant_id
    ].copy()
    wb_p = df_wakeup_bedtime.loc[
        df_wakeup_bedtime['ParticipantIdentifier'] == participant_id
    ].copy()

    if hr_p.empty:
        continue

    hr_p['Date'] = pd.to_datetime(hr_p['Date']).dt.normalize()
    step_p['Date'] = pd.to_datetime(step_p['Date']).dt.normalize()

    min_date = pd.to_datetime(
        summary_surveytask.loc[
            summary_surveytask['ParticipantIdentifier'] == participant_id
        ].iloc[0].date_min,
        errors='coerce'
    )
    if pd.isna(min_date):
        continue

    min_date = min_date.normalize()
    date_range_length = 85

    for i in range(date_range_length):
        date = min_date + pd.Timedelta(days=i)
        wakeup_time, bedtime_time = _get_wakeup_bedtime(wb_p, date)
        if wakeup_time is None or bedtime_time is None:
            continue

        wakeup_dt = pd.Timestamp.combine(date, wakeup_time)
        morning_end = wakeup_dt + pd.Timedelta(hours=1)
        bedtime_dt = pd.Timestamp.combine(date, bedtime_time)

        # Rest-of-day: after wakeup+1 hour to bedtime
        hr_rest = hr_p.loc[
            (hr_p['DateTime'] >= morning_end) &
            (hr_p['DateTime'] <= bedtime_dt)
        ]
        step_rest = step_p.loc[
            (step_p['DateTime'] >= morning_end) &
            (step_p['DateTime'] <= bedtime_dt)
        ]

        restday_valid_minutes = _valid_minutes(hr_rest, step_rest)
        morning_wearing = int(morning_lookup.get((participant_id, date), 0))

        wear_rows.append({
            'ParticipantIdentifier': participant_id,
            'Date': date,
            'restday_valid_minutes': restday_valid_minutes,
            'morning_wearing': morning_wearing,
            'restday_wearing': int(restday_valid_minutes > RESTDAY_MIN_VALID_MINUTES)
        })

wear_day = pd.DataFrame(wear_rows)

wear_2x2 = pd.crosstab(
    wear_day['morning_wearing'].astype(int),
    wear_day['restday_wearing'].astype(int),
    rownames=['Morning wearing (0/1)'],
    colnames=['Rest-of-day wearing (0/1)'],
    dropna=False
).reindex(index=[0, 1], columns=[0, 1], fill_value=0)

print('2x2 count table:')
print(wear_2x2)

print('\n2x2 proportion table:')
print((wear_2x2 / wear_2x2.to_numpy().sum()).round(4))

# save the next morning wearing data
wear_day.to_csv(folder/ 'wear_day.csv', index=False)

2x2 count table:
Rest-of-day wearing (0/1)   0    1
Morning wearing (0/1)             
0                          93   71
1                           4  767

2x2 proportion table:
Rest-of-day wearing (0/1)       0       1
Morning wearing (0/1)                    
0                          0.0995  0.0759
1                          0.0043  0.8203


In [256]:
# extract hourly step counts in between wakeup and bedtime

hourly_step_counts = []

for participant_id in complete_participant_ids:
    stepcountbymin_participant = stepcountbymin_selected.loc[stepcountbymin_selected['ParticipantIdentifier'] == participant_id].copy()
    df_wakeup_bedtime_participant = df_wakeup_bedtime.loc[df_wakeup_bedtime['ParticipantIdentifier'] == participant_id].copy()

    min_date = summary_surveytask.loc[
        summary_surveytask['ParticipantIdentifier'] == participant_id
    ].iloc[0].date_min - pd.Timedelta(days=7)
    # print(min_date)
    date_range_length = 92

    for i in range(date_range_length):
        date = min_date + pd.Timedelta(days=i)
        
        if df_wakeup_bedtime_participant.shape[0] == 1:
            weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[0]
            weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[0]
            weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[0]
            weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[0]
        else:
            change_date = df_wakeup_bedtime_participant['QuestionEndDate']
            weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[0]
            weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[0]
            weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[0]
            weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[0]
            
            for j in range(len(df_wakeup_bedtime_participant)):
                change_date = df_wakeup_bedtime_participant['QuestionEndDate'].iloc[j]
                
                # Check if this is the applicable period
                if j < len(df_wakeup_bedtime_participant) - 1:
                    next_change_date = df_wakeup_bedtime_participant['QuestionEndDate'].iloc[j + 1]
                    if change_date <= date < next_change_date:
                        weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[j]
                        weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[j]
                        weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[j]
                        weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[j]
                        break
                else:
                    # Last entry - applies from change_date onwards
                    if change_date <= date:
                        weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[j]
                        weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[j]
                        weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[j]
                        weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[j]
                        break
        
        # Determine wakeup time based on weekday/weekend
        is_weekday = date.weekday() < 5
        wakeup_time = weekday_wakeup if is_weekday else weekend_wakeup
        bedtime_time = weekday_bedtime if is_weekday else weekend_bedtime

        # print(participant_id)
        # print(date, wakeup_time, bedtime_time)

        # wake up set to floor
        # anchor_date = pd.Timestamp('2000-01-01')  
        # wakeup_floor = (
        # pd.Timestamp.combine(anchor_date, wakeup_time)
        # .floor('h')                       
        # .time()
        # )

        # bedtime set to ceil
        # bedtime_ceil = (
        # pd.Timestamp.combine(anchor_date, bedtime_time)
        # .ceil('h')                       
        # .time()
        # )

        # Get all heart rate data for this date
        stepcountbymin_participant_date = stepcountbymin_participant[stepcountbymin_participant.Date == date]
        stepcountbymin_participant_date = stepcountbymin_participant_date[stepcountbymin_participant_date.DateTime.dt.time >= wakeup_time]
        stepcountbymin_participant_date = stepcountbymin_participant_date[stepcountbymin_participant_date.DateTime.dt.time <= bedtime_time]
        
        # Create hourly bins from wakeup to bedtime
        wakeup_datetime = pd.Timestamp.combine(date, wakeup_time)
        bedtime_datetime = pd.Timestamp.combine(date, bedtime_time)
        

        
        # Generate hourly time bins
        decision_time = 0
        num_decisions = 2 # 2 decision points per day wakeup + 1, wakeup + 6
        while decision_time < num_decisions:
            start_window = wakeup_datetime + pd.Timedelta(hours=1) if decision_time == 0 else wakeup_datetime + pd.Timedelta(hours=6)
            end_window = start_window + pd.Timedelta(hours=4)

            # read heart rate data for this hour
            missing_hours_participant_date = df_missing_hours[
                (df_missing_hours['ParticipantIdentifier'] == participant_id) &
                (df_missing_hours['Date'] == date) &
                (df_missing_hours['DecisionTime'] == decision_time)
            ]

            wearing = missing_hours_participant_date['HourWearing'].iloc[0]
            
            # Filter data for this hour
            hour_data = stepcountbymin_participant_date[
                (stepcountbymin_participant_date['DateTime'] >= start_window) &
                (stepcountbymin_participant_date['DateTime'] < end_window)
            ]
            
            # Filter valid step count readings (not 0 and not NaN)
            valid_sc_init = hour_data[
                (hour_data['Value'].notna())
            ]
            
            # Calculate number of valid step counts in this hour
            if (len(valid_sc_init) > 0) and (wearing == 1):
                valid_sc = sum(valid_sc_init['Value'])
            else:
                valid_sc = np.nan

            # print the proportion of wearing == 1 but valid_sc !=0
            if len(valid_sc_init) > 0:
                check_valid_sc = sum(valid_sc_init['Value'])
                check_status = int((check_valid_sc > 0) & (wearing == 0))
            else:
                check_status = np.nan


            hourly_step_counts.append({
                'ParticipantIdentifier': participant_id,
                'Date': date,
                'DecisionTime': decision_time,
                'DateTimeStart': start_window,
                'StepCount': valid_sc,
                'CheckStatus': check_status
            })
            
            decision_time += 1

df_hourly_step_counts = pd.DataFrame(hourly_step_counts)

# add exponential moving average of step count
df_hourly_step_counts['EMA_StepCount'] = df_hourly_step_counts['StepCount'].ewm(span=7, adjust=False).mean()

# print(df_hourly_step_counts[df_hourly_step_counts['participantidentifier'] == 22])
# print((df_hourly_step_counts[df_hourly_step_counts['ParticipantIdentifier'] == 31]))

# print the proportion of wearing == 0 but valid_sc !=0
print(len(df_hourly_step_counts[df_hourly_step_counts['CheckStatus'] == 1]) / len(df_hourly_step_counts))
nan_ratio = (
    df_hourly_step_counts['CheckStatus'].isna().sum()
    / len(df_hourly_step_counts)
)
print(nan_ratio)

0.07361660079051384
0.0691699604743083


In [257]:
print(df_hourly_step_counts[df_hourly_step_counts['ParticipantIdentifier'] == "219"][101:150])

Empty DataFrame
Columns: [ParticipantIdentifier, Date, DecisionTime, DateTimeStart, StepCount, CheckStatus, EMA_StepCount]
Index: []


In [258]:
yesterday_step_counts = []

for participant_id in complete_participant_ids:
    stepcountbymin_participant = stepcountbymin_selected.loc[stepcountbymin_selected['ParticipantIdentifier'] == participant_id].copy()
    df_wakeup_bedtime_participant = df_wakeup_bedtime.loc[df_wakeup_bedtime['ParticipantIdentifier'] == participant_id].copy()

    min_date = summary_surveytask.loc[
        summary_surveytask['ParticipantIdentifier'] == participant_id
    ].iloc[0].date_min - pd.Timedelta(days=7)
    # print(min_date)
    date_range_length = 92

    for i in range(date_range_length):
        date = min_date + pd.Timedelta(days=i)
        
        if df_wakeup_bedtime_participant.shape[0] == 1:
            weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[0]
            weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[0]
            weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[0]
            weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[0]
        else:
            change_date = df_wakeup_bedtime_participant['QuestionEndDate']
            weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[0]
            weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[0]
            weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[0]
            weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[0]
            
            for j in range(len(df_wakeup_bedtime_participant)):
                change_date = df_wakeup_bedtime_participant['QuestionEndDate'].iloc[j]
                
                # Check if this is the applicable period
                if j < len(df_wakeup_bedtime_participant) - 1:
                    next_change_date = df_wakeup_bedtime_participant['QuestionEndDate'].iloc[j + 1]
                    if change_date <= date < next_change_date:
                        weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[j]
                        weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[j]
                        weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[j]
                        weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[j]
                        break
                else:
                    # Last entry - applies from change_date onwards
                    if change_date <= date:
                        weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[j]
                        weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[j]
                        weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[j]
                        weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[j]
                        break
        
        # Determine wakeup time based on weekday/weekend
        is_weekday = date.weekday() < 5
        wakeup_time = weekday_wakeup if is_weekday else weekend_wakeup
        bedtime_time = weekday_bedtime if is_weekday else weekend_bedtime


        # Get all heart rate data for this date
        stepcountbymin_participant_date = stepcountbymin_participant[stepcountbymin_participant.Date == date]
 
        # Create hourly bins from wakeup to bedtime
        wakeup_datetime = pd.Timestamp.combine(date, wakeup_time)
        bedtime_datetime = pd.Timestamp.combine(date, bedtime_time)
        

        
 
            # read heart rate data for this hour
        missing_days_participant_date = df_missing_days[
            (df_missing_days['ParticipantIdentifier'] == participant_id) &
            (df_missing_days['Date'] == date)
        ]

        wearing = missing_days_participant_date['DayWearing'].iloc[0]
            
            # Filter data for this hour
        daily_data = stepcountbymin_participant_date[
            (stepcountbymin_participant_date['DateTime'] >= wakeup_datetime) &
            (stepcountbymin_participant_date['DateTime'] < bedtime_datetime)
        ]
            
            # Filter valid step count readings (not 0 and not NaN)
        valid_sc = daily_data[
            (daily_data['Value'].notna())
        ]
            
        # 
            # Calculate number of valid step counts in this hour
        if (len(valid_sc) > 0) and (wearing == 1):
            valid_sc = sum(valid_sc['Value'])
        else:
            valid_sc = np.nan
            

        yesterday_step_counts.append({
            'ParticipantIdentifier': participant_id,
            'Date': date + pd.Timedelta(days=1),
            'YesterdayStepCount': valid_sc
        })
        

df_yesterday_step_counts = pd.DataFrame(yesterday_step_counts)

#print ratio of wearing == 1 but valid_sc !=0
print(len(df_yesterday_step_counts[df_yesterday_step_counts['YesterdayStepCount'] > 0]) / len(df_yesterday_step_counts))

print(df_yesterday_step_counts[df_yesterday_step_counts['ParticipantIdentifier'] == "151"])

#print rate of nan yesterday step counts
print(len(df_yesterday_step_counts[df_yesterday_step_counts['YesterdayStepCount'].isna()]) / len(df_yesterday_step_counts))


0.8814229249011858
    ParticipantIdentifier        Date  YesterdayStepCount
276                   151  2025-09-23             16785.0
277                   151  2025-09-24             14451.0
278                   151  2025-09-25              6106.0
279                   151  2025-09-26             12174.0
280                   151  2025-09-27             24695.0
..                    ...         ...                 ...
363                   151  2025-12-19                 NaN
364                   151  2025-12-20                 NaN
365                   151  2025-12-21                 NaN
366                   151  2025-12-22                 NaN
367                   151  2025-12-23                 NaN

[92 rows x 3 columns]
0.11857707509881422


In [259]:
# extract hourly step counts in between wakeup and bedtime

prior_2hours_step_counts = []

for participant_id in complete_participant_ids:
    stepcountbymin_participant = stepcountbymin_selected.loc[stepcountbymin_selected['ParticipantIdentifier'] == participant_id].copy()
    df_wakeup_bedtime_participant = df_wakeup_bedtime.loc[df_wakeup_bedtime['ParticipantIdentifier'] == participant_id].copy()

    min_date = summary_surveytask.loc[
        summary_surveytask['ParticipantIdentifier'] == participant_id
    ].iloc[0].date_min
    # print(min_date)
    date_range_length = 85

    for i in range(date_range_length):
        date = min_date + pd.Timedelta(days=i)
        
        if df_wakeup_bedtime_participant.shape[0] == 1:
            weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[0]
            weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[0]
            weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[0]
            weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[0]
        else:
            change_date = df_wakeup_bedtime_participant['QuestionEndDate']
            weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[0]
            weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[0]
            weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[0]
            weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[0]
            
            for j in range(len(df_wakeup_bedtime_participant)):
                change_date = df_wakeup_bedtime_participant['QuestionEndDate'].iloc[j]
                
                # Check if this is the applicable period
                if j < len(df_wakeup_bedtime_participant) - 1:
                    next_change_date = df_wakeup_bedtime_participant['QuestionEndDate'].iloc[j + 1]
                    if change_date <= date < next_change_date:
                        weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[j]
                        weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[j]
                        weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[j]
                        weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[j]
                        break
                else:
                    # Last entry - applies from change_date onwards
                    if change_date <= date:
                        weekday_wakeup = df_wakeup_bedtime_participant['WeekdayWakeup'].iloc[j]
                        weekend_wakeup = df_wakeup_bedtime_participant['WeekendWakeup'].iloc[j]
                        weekday_bedtime = df_wakeup_bedtime_participant['WeekdayBedtime'].iloc[j]
                        weekend_bedtime = df_wakeup_bedtime_participant['WeekendBedtime'].iloc[j]
                        break
        
        # Determine wakeup time based on weekday/weekend
        is_weekday = date.weekday() < 5
        wakeup_time = weekday_wakeup if is_weekday else weekend_wakeup
        bedtime_time = weekday_bedtime if is_weekday else weekend_bedtime


        # Get all heart rate data for this date
        stepcountbymin_participant_date = stepcountbymin_participant[stepcountbymin_participant.Date == date]
        stepcountbymin_participant_date = stepcountbymin_participant_date[stepcountbymin_participant_date.DateTime.dt.time >= wakeup_time]
        stepcountbymin_participant_date = stepcountbymin_participant_date[stepcountbymin_participant_date.DateTime.dt.time <= bedtime_time]
        
        # Create hourly bins from wakeup to bedtime
        wakeup_datetime = pd.Timestamp.combine(date, wakeup_time)
        bedtime_datetime = pd.Timestamp.combine(date, bedtime_time)
        

        
        # Generate hourly time bins
        decision_time = 0
        num_decisions = 2 # 2 decision points per day wakeup + 1, wakeup + 6
        while decision_time < num_decisions:
            end_window = wakeup_datetime + pd.Timedelta(hours=1) if decision_time == 0 else wakeup_datetime + pd.Timedelta(hours=6)
            start_window = end_window - pd.Timedelta(hours=0.5)

            # read heart rate data for this hour
            missing_2hours_participant_date = df_2hours[
                (df_2hours['ParticipantIdentifier'] == participant_id) &
                (df_2hours['Date'] == date) &
                (df_2hours['DecisionTime'] == decision_time)
            ]

            wearing = missing_2hours_participant_date['HourWearing'].iloc[0]
            
            # Filter data for this hour
            hour_data = stepcountbymin_participant_date[
                (stepcountbymin_participant_date['DateTime'] >= start_window) &
                (stepcountbymin_participant_date['DateTime'] < end_window)
            ]
            
            # Filter valid step count readings (not 0 and not NaN)
            valid_sc_init = hour_data[
                (hour_data['Value'].notna())
            ]
            
            # Calculate number of valid step counts in this hour
            if (len(valid_sc_init) > 0) and (wearing == 1):
                valid_sc = sum(valid_sc_init['Value'])
            else:
                valid_sc = np.nan

            # print the proportion of wearing == 1 but valid_sc !=0
            if len(valid_sc_init) > 0:
                check_valid_sc = sum(valid_sc_init['Value'])
                check_status = int((check_valid_sc > 0) & (wearing == 0))
            else:
                check_status = np.nan


            prior_2hours_step_counts.append({
                'ParticipantIdentifier': participant_id,
                'Date': date,
                'DecisionTime': decision_time,
                # 'DateTimeStart': start_window,
                'StepCount': valid_sc,
                'CheckStatus': check_status
            })
            
            decision_time += 1

df_prior_2hours_step_counts = pd.DataFrame(prior_2hours_step_counts)


# print(df_hourly_step_counts[df_hourly_step_counts['participantidentifier'] == 22])
print((df_prior_2hours_step_counts[df_prior_2hours_step_counts['ParticipantIdentifier'] == 31]))

# print the proportion of wearing == 1 but valid_sc !=0
print(len(df_prior_2hours_step_counts[df_prior_2hours_step_counts['CheckStatus'] == 1]) / len(df_prior_2hours_step_counts))
nan_ratio = (
    df_prior_2hours_step_counts['CheckStatus'].isna().sum()
    / len(df_prior_2hours_step_counts)
)
print(nan_ratio)

Empty DataFrame
Columns: [ParticipantIdentifier, Date, DecisionTime, StepCount, CheckStatus]
Index: []
0.0481283422459893
0.07058823529411765


In [260]:
# save the dataframe
df_yesterday_step_counts.to_csv(os.path.join(folder, 'yesterday_step_counts.csv'), index=False)
df_hourly_step_counts.to_csv(os.path.join(folder, 'hourly_step_counts.csv'), index=False)
df_prior_2hours_step_counts.to_csv(os.path.join(folder, 'prior_2hours_step_counts.csv'), index=False)

In [261]:
# Recorded physical activity data
fitbit_log_data = pd.read_csv(folder / 'FitbitActivityLogs.csv')
fitbit_log_data['Date'] = pd.to_datetime(fitbit_log_data['EndDate']).dt.date
recorded_physical_activity = []
for participant_id in complete_participant_ids:
    fitbit_log_participant = fitbit_log_data.loc[fitbit_log_data['ParticipantIdentifier'] == participant_id].copy()

    min_date = summary_surveytask.loc[
        summary_surveytask['ParticipantIdentifier'] == participant_id
    ].iloc[0].date_min - pd.Timedelta(days=7)
    # print(min_date)
    date_range_length = 92

    for i in range(date_range_length):
        date = min_date + pd.Timedelta(days=i)

        day_log = fitbit_log_participant[fitbit_log_participant["Date"] == date]
        if day_log.empty:
            rpa = 0
        else:
            types = day_log["ActivityName"].dropna().unique()
            rpa = 0 if (len(types) == 1 and types[0] == "Walk") else 1

        recorded_physical_activity.append({
            "ParticipantIdentifier": participant_id,
            "Date": date,
            "RecordedPhysicalActivity": rpa,
        })

df_recorded_physical_activity = pd.DataFrame(recorded_physical_activity)

# add a column of previous 7 days fraction of recorded physical activity
df_recorded_physical_activity['Previous7DaysRPA'] = df_recorded_physical_activity.groupby('ParticipantIdentifier')['RecordedPhysicalActivity'].transform(lambda x: x.shift(1).rolling(window=7).mean())

print(df_recorded_physical_activity)


#save the dataframe
df_recorded_physical_activity.to_csv(os.path.join(folder, 'recorded_physical_activity.csv'), index=False)


     ParticipantIdentifier        Date  RecordedPhysicalActivity  \
0                      118  2025-09-07                         0   
1                      118  2025-09-08                         0   
2                      118  2025-09-09                         0   
3                      118  2025-09-10                         0   
4                      118  2025-09-11                         0   
...                    ...         ...                       ...   
1007                   225  2026-03-05                         0   
1008                   225  2026-03-06                         0   
1009                   225  2026-03-07                         0   
1010                   225  2026-03-08                         1   
1011                   225  2026-03-09                         0   

      Previous7DaysRPA  
0                  NaN  
1                  NaN  
2                  NaN  
3                  NaN  
4                  NaN  
...                ...  
1007    

In [262]:
# Active minutes data